# Flexibility needs and contribution 
**sources:**
- https://www.artelys.com/app/docs/supergrid/kpiDoc.html#flexibility-needs-wh
- https://www.artelys.com/app/docs/supergrid/kpiDoc.html#contribution-to-flexibility-needs-wh


### imports and definitions


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa
from utils import (
    carriers_in_german,
    flex_techs,
    flex_techs_demand,
    flex_techs_supply,
    tech_colors,
)

In [ ]:
all_flex_techs = list(flex_techs_supply.keys()) + list(flex_techs_demand.keys())

year_colors = {
    2020: "dimgrey",
    2025: "darkorange",
    2030: "seagreen",
    2035: "cadetblue",
    2040: "hotpink",
    2045: "darkviolet",
}
load_carriers = ["electricity", "agriculture electricity", "industry electricity"]
vre_gens = [
    "onwind",
    "offwind-ac",
    "offwind-dc",
    "solar",
    "solar-hsat",
    "solar rooftop",
    "ror",
]

plt.rcParams["font.family"] = "Arial"

### functions

In [ ]:
def summarize_small_values(df, column_name, threshold_percentage=3):
    total_sum = df[column_name].sum()
    threshold = (threshold_percentage / 100) * total_sum
    small_values = df[df[column_name] < threshold]
    other_sum = small_values[column_name].sum()
    df = df[df[column_name] >= threshold]
    other_row = pd.DataFrame({column_name: [other_sum]}, index=["Other"])
    df = pd.concat([df, other_row])
    return df

In [ ]:
def supply_demand(
    n,
    region="DE",
    interconnectors=True,
    merge_dist_grid=True,
    drop_dist_grid=True,
    buses=None,
    add_diff_as_import=False,
):
    """
    Aggregates electricity supply and demand statistics from a PyPSA network for a given region.

    Parameters
    ----------
    n : pypsa.Network
        The PyPSA network object containing snapshots, buses, and generators.
    region : str, default "DE"
        Two-letter country code prefix for buses to include.
    interconnectors : bool, default True
        Whether to include interconnectors (AC/DC) in the supply/demand.
    merge_dist_grid : bool, default True
        If True, merges low-voltage and AC buses so that the distribution grid only represents losses.
    drop_dist_grid : bool, default True
        If True, removes the electricity distribution grid from supply and demand.
    buses : list[str] or None
        Specific bus names to consider; defaults to all buses in the given region with low-voltage or AC carrier.
    add_diff_as_import : bool, default False
        If True, any difference between total supply and demand is added as "import" (negative difference) or "export" (positive difference).

    Returns
    -------
    electricity_supply : pd.DataFrame
        Supply per carrier aggregated over the selected buses.
    electricity_demand : pd.DataFrame
        Demand per carrier aggregated over the selected buses.

    Notes
    -----
    - Uses n.statistics.supply and n.statistics.withdrawal to compute per-bus and per-carrier values.
    - Multiplies by n.snapshot_weightings.generators to scale per snapshot.
    - Can remove AC/DC interconnectors or the distribution grid according to flags.
    - Optionally computes residual import/export if add_diff_as_import=True.
    - Prints total system supply and demand in TWh (sum / 1e6).
    """

    bus_carrier = ["low voltage", "AC"]

    kwargs = {
        "groupby": n.statistics.groupers.get_name_bus_and_carrier,
        "nice_names": False,
    }

    electricity_supply = (
        n.statistics.supply(bus_carrier=bus_carrier, aggregate_time=False, **kwargs)
    ).multiply(n.snapshot_weightings.generators)

    if buses is None:
        buses = n.buses[
            (n.buses.index.str[:2] == region) & (n.buses.carrier.isin(bus_carrier))
        ].index

    electricity_supply = (
        electricity_supply[electricity_supply.index.get_level_values("bus").isin(buses)]
        .groupby("carrier")
        .sum()
    )

    electricity_demand = (
        n.statistics.withdrawal(bus_carrier=bus_carrier, aggregate_time=False, **kwargs)
    ).multiply(n.snapshot_weightings.generators)
    electricity_demand = (
        electricity_demand[electricity_demand.index.get_level_values("bus").isin(buses)]
        .groupby("carrier")
        .sum()
    )

    if not interconnectors:
        electricity_supply = electricity_supply.drop(["AC", "DC"], errors="ignore")
        electricity_demand = electricity_demand.drop(["AC", "DC"], errors="ignore")

    if merge_dist_grid:
        # merge AC & low voltage such that electricity distribution grid does only function as a demand representing the grid losses
        electricity_demand.loc["electricity distribution grid losses", :] = abs(
            electricity_supply.loc["electricity distribution grid"]
            - electricity_demand.loc["electricity distribution grid"]
        )
        electricity_demand = electricity_demand.drop(
            "electricity distribution grid", axis=0
        )
        electricity_supply = electricity_supply.drop(
            "electricity distribution grid", axis=0
        )

    if drop_dist_grid:
        electricity_supply = electricity_supply.drop(
            "electricity distribution grid", errors="ignore"
        )
        electricity_demand = electricity_demand.drop(
            "electricity distribution grid", errors="ignore"
        )

    if add_diff_as_import:
        diff = electricity_supply.sum() - electricity_demand.sum()
        electricity_supply.loc["import", :] = diff.clip(upper=0).abs()
        electricity_demand.loc["export", :] = diff.clip(lower=0)

    # # only keep rows with minimum of 1 MW
    # electricity_supply = electricity_supply[electricity_supply.sum(axis=1) > 1]
    # electricity_demand = electricity_demand[electricity_demand.sum(axis=1) > 1]

    print(electricity_supply.sum().sum() / 1e6)
    print(electricity_demand.sum().sum() / 1e6)

    return electricity_supply, electricity_demand

### loading data & tech colors

In [ ]:
# networks_old = {}
# for year in np.arange(2020,2050,5):
#     fn = f"/home/julian-geis/Documents/04_Ariadne/run_results/20241108-limit-ft-meoh/KN2045_Bal_v4/postnetworks/base_s_49_lvopt__none_{year}.nc"
#     networks_old[year] = pypsa.Network(fn)

run = "20250214-reworkimportban"  # "20241203-force-onwind-south"
scenario = "KN2045_Elec_v4"  # "KN2045_H2_v4" # "CurrentPolicies" # "KN2045_Bal_v4"
weather_year = 2010  # 1996 # 2010

networks = {}

for year in np.arange(2020, 2050, 5):
    fn = f"/home/julian-geis/Documents/04_Ariadne/run_results/{run}/{scenario}/postnetworks/base_s_49_lvopt__none_{year}.nc"
    networks[year] = pypsa.Network(fn)

# # from z1
# for year in np.arange(2020, 2050, 5):
#     fn = f"/home/julian-geis/cluster/z1/pypsa-de/results/20250604_weather_sensitivities/Mix{weather_year}/networks/base_s_49__none_{year}.nc"
#     networks[year] = pypsa.Network(fn)


PLOT_DIR = "/home/julian-geis/Documents/06_PhD/02Flexibility/plots/" + run

## supply and demand

In [ ]:
# Electricity supply
# electricity generation DE 2020: 488 TWh
# if aggregate_time=False the results are in MW (you need to multiply with snapshot.weightings)

region = "DE"
n = networks[2045]

kwargs = {
    "groupby": n.statistics.groupers.get_name_bus_and_carrier,
    "nice_names": False,
}

electricity_supply_de = (
    n.statistics.supply(bus_carrier=["low voltage", "AC"], **kwargs)
    .filter(like=region)
    .groupby(["carrier"])
    .sum()
    .drop(
        ["AC", "DC", "electricity distribution grid"],
        errors="ignore",
    )
)

print(round(electricity_supply_de.sum() / 1e6, 2))

In [ ]:
# real supply without Storage
electricity_supply_de[
    ~electricity_supply_de.index.str.contains(
        "PHS|battery discharger|home battery discharger|V2G"
    )
].sum() / 1e6

In [ ]:
# suppy from VRE
electricity_supply_de[vre_gens].sum() / 1e6

In [ ]:
# supply from storage
storage_gens = ["PHS", "battery discharger", "home battery discharger"]
electricity_supply_de[storage_gens].sum() / 1e6

In [ ]:
# non-VRE supply
non_vre_gens = electricity_supply_de[
    ~electricity_supply_de.index.isin(vre_gens + storage_gens)
].index
electricity_supply_de[non_vre_gens].sum() / 1e6

## Backup capacity and generation

In [ ]:
df_all = pd.DataFrame()

for year in np.arange(2020, 2050, 5):
    n = networks[year]

    electricity_cap = (
        n.statistics.optimal_capacity(bus_carrier=["low voltage", "AC"], **kwargs)
        .filter(like=region)
        .groupby(["carrier"])
        .sum()
        .drop(
            ["AC", "DC", "electricity distribution grid"],
            errors="ignore",
        )
    )

    df = round(electricity_cap.sort_values(ascending=False) / 1e3, 2)
    # non_vre_gens = df[~df.index.isin(vre_gens)].index
    # df = df.loc[non_vre_gens]
    df = df[df > 1e-3]

    df_all = pd.concat([df_all, df], axis=1)

df_all.columns = np.arange(2020, 2050, 5)

In [ ]:
backup_techs = {
    "Gas": ["OCGT", "CCGT", "urban central gas CHP", "urban central gas CHP CC"],
    # "Öl": ["oil", "urban central oil CHP"],
    "Kohle": ["lignite", "coal", "urban central coal CHP", "urban central lignite CHP"],
    "Wasserkraft": ["PHS", "hydro", "ror"],
    "Batterie": ["battery discharger", "home battery discharger"],
    "Wasserstoff": [
        "H2 OCGT",
        "H2 retrofit OCGT",
        "urban central H2 CHP",
        "urban central H2 retrofit CHP",
    ],
    "Sonstige": [
        "solid biomass",
        "urban central solid biomass CHP",
        "urban central solid biomass CHP CC",
        "biogas",
        "waste CHP",
        "waste CHP CC",
    ],
}

In [ ]:
df_all

In [ ]:
# Create figure
plt.figure(figsize=(18, 4))

# Track x-axis positions
x_positions = []
x_labels = []
x_offset = 0

df = df_all

# Plot for each group
for group, techs_in_group in backup_techs.items():
    # Find matching techs in the dataframe
    matching_techs = [tech for tech in techs_in_group if tech in df.index]

    # Prepare data for this group
    group_data = df.loc[matching_techs]

    # Prepare bottom for stacking
    bottom = np.zeros(len(group_data.columns))

    # Plot each technology in the group
    for j, tech in enumerate(matching_techs):
        values = group_data.loc[tech].fillna(0)
        plt.bar(
            np.arange(len(group_data.columns)) + x_offset,
            values,
            1,
            bottom=bottom,
            label=tech,
            color=tech_colors[tech],
        )
        bottom += values

    # Store x-axis positions for this group
    x_positions.append(x_offset + len(group_data.columns) / 2)
    x_labels.append(group)

    # Update x offset with extra spacing
    x_offset += len(group_data.columns) + 1  # Add extra space between groups

plt.ylim(0, 120)

# Customize the plot
plt.title("Kapazität Backup-Kraftwerke (Strom) [GW]", fontsize=16)
plt.ylabel("GW", fontsize=16)

# Create custom x-tick labels with group names below years
x_ticks = np.concatenate(
    [np.arange(len(df.columns)) + i for i in range(0, x_offset, len(df.columns) + 1)]
)
x_tick_labels = np.tile(df.columns, len(backup_techs))

plt.xticks(x_ticks, x_tick_labels, rotation=45)

# Add group labels below x-axis ticks
for pos, label in zip(x_positions, x_labels):
    plt.text(
        pos,
        plt.gca().get_ylim()[0] - plt.gca().get_ylim()[1] * 0.15,
        label,
        horizontalalignment="center",
        verticalalignment="top",
        fontweight="bold",
        fontsize=16,
    )

plt.grid(axis="y")

# Replace legend labels with German names from carriers_in_german
handles, labels = plt.gca().get_legend_handles_labels()
new_labels = [
    carriers_in_german.get(label, label) for label in labels
]  # Replace labels if in dict
plt.legend(handles, new_labels, loc="upper center", ncol=7)

plt.tight_layout()

plt.savefig(PLOT_DIR + "/backup_capacity.png")
plt.savefig(PLOT_DIR + "/backup_capacity.pdf")

In [ ]:
region = "DE"

kwargs = {
    "groupby": n.statistics.groupers.get_name_bus_and_carrier,
    "nice_names": False,
}

df_all = pd.DataFrame()

for year in np.arange(2020, 2050, 5):
    n = networks[year]

    electricity_supply_de = (
        n.statistics.supply(bus_carrier=["low voltage", "AC"], **kwargs)
        .filter(like=region)
        .groupby(["carrier"])
        .sum()
        .drop(
            ["AC", "DC", "electricity distribution grid"],
            errors="ignore",
        )
    )

    df = round(electricity_supply_de.sort_values(ascending=False) / 1e6, 2)
    non_vre_gens = df[~df.index.isin(vre_gens)].index
    df = df.loc[non_vre_gens]
    df = df[df > 0.01]
    df_all = pd.concat([df_all, df], axis=1)
df_all.columns = np.arange(2020, 2050, 5)

In [ ]:
df_all

In [ ]:
# Create figure
plt.figure(figsize=(18, 5))

# Track x-axis positions
x_positions = []
x_labels = []
x_offset = 0

df = df_all

# Plot for each group
for group, techs_in_group in backup_techs.items():
    # Find matching techs in the dataframe

    matching_techs = [tech for tech in techs_in_group if tech in df.index]

    # Prepare data for this group
    group_data = df.loc[matching_techs]

    # Prepare bottom for stacking
    bottom = np.zeros(len(group_data.columns))

    # Plot each technology in the group
    for j, tech in enumerate(matching_techs):
        values = group_data.loc[tech].fillna(0)
        plt.bar(
            np.arange(len(group_data.columns)) + x_offset,
            values,
            1,
            bottom=bottom,
            label=tech,
            color=tech_colors[tech],
        )
        bottom += values

    # Store x-axis positions for this group
    x_positions.append(x_offset + len(group_data.columns) / 2)
    x_labels.append(group)

    # Update x offset with extra spacing
    x_offset += len(group_data.columns) + 1  # Add extra space between groups

plt.ylim(0, 200)

# Customize the plot
plt.title("Versorgung Backup-Kraftwerke (Strom) [TWh]", fontsize=24)
plt.ylabel("TWh", fontsize=16)

# Create custom x-tick labels with group names below years
x_ticks = np.concatenate(
    [np.arange(len(df.columns)) + i for i in range(0, x_offset, len(df.columns) + 1)]
)
x_tick_labels = np.tile(df.columns, len(backup_techs))

plt.xticks(x_ticks, x_tick_labels, rotation=45)

# Add group labels below x-axis ticks
for pos, label in zip(x_positions, x_labels):
    plt.text(
        pos,
        plt.gca().get_ylim()[0] - plt.gca().get_ylim()[1] * 0.15,
        label,
        horizontalalignment="center",
        verticalalignment="top",
        fontweight="bold",
        fontsize=16,
    )

plt.grid(axis="y")

# Replace legend labels with German names from carriers_in_german
handles, labels = plt.gca().get_legend_handles_labels()
new_labels = [
    carriers_in_german.get(label, label) for label in labels
]  # Replace labels if in dict
plt.legend(handles, new_labels, loc="upper center", ncol=7)

plt.tight_layout()

plt.savefig(PLOT_DIR + "/backup_dispatch.png")
plt.savefig(PLOT_DIR + "/backup_dispatch.pdf")

In [ ]:
n = networks[2045]
s, d = supply_demand(
    n, interconnectors=False, merge_dist_grid=False, drop_dist_grid=False
)

In [ ]:
df = s.T.resample("ME").sum() / 1e6
df.index = df.index.strftime("%b")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

df[non_vre_gens].plot(
    kind="bar", stacked=True, ax=ax, color=[tech_colors[tech] for tech in non_vre_gens]
)


plt.title(
    "Strombereitstellung Backup-Kraftwerke und Speicher 2045"
)  # Add the desired title
plt.ylabel("TWh")  # Add the desired Y-axis label
plt.xticks(rotation=0)  # 0 degrees for horizontal labels

# Customize legend and axis limits
plt.legend(bbox_to_anchor=[1.22, 0.5], loc="center")
plt.ylim(0, 20)

# Show the plot
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Translate the non-VRE generators using the dictionary
translated_labels = [carriers_in_german.get(tech, tech) for tech in non_vre_gens]

# Plot the data with translated labels
df[non_vre_gens].plot(
    kind="bar", stacked=True, ax=ax, color=[tech_colors[tech] for tech in non_vre_gens]
)

# Add title and labels
plt.title(
    "Strombereitstellung Backup-Kraftwerke und Speicher 2045"
)  # Add the desired title
plt.ylabel("TWh")  # Add the desired Y-axis label
plt.xticks(rotation=0)  # 0 degrees for horizontal labels

# Customize legend and axis limits
plt.legend(
    translated_labels, bbox_to_anchor=[1.22, 0.5], loc="center"
)  # Use translated labels for legend
plt.ylim(0, 20)

# Show the plot
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

tech_colors["battery discharger"] = "#61ba70"

# Calculate the total contribution for each carrier
total_contributions = df[non_vre_gens].sum()

# Identify carriers contributing less than 1%
threshold = total_contributions.sum() * 0.001
small_contributors = total_contributions[total_contributions < threshold].index

# Create a new DataFrame with small contributors combined as "Sonstige"
df_combined = df[non_vre_gens].copy()
df_combined["Sonstige"] = df_combined[small_contributors].sum(axis=1)
df_combined = df_combined.drop(columns=small_contributors)

# Translate labels using the dictionary, adding "Sonstige"
translated_labels = [carriers_in_german.get(tech, tech) for tech in df_combined.columns]

# Plot the data
df_combined.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=[tech_colors.get(tech, "gray") for tech in df_combined.columns],
)

# Add title and labels
plt.title(
    "Strombereitstellung Backup-Kraftwerke und Speicher 2045"
)  # Add the desired title
plt.ylabel("TWh")  # Add the desired Y-axis label
plt.xticks(rotation=0)  # 0 degrees for horizontal labels

# Customize legend and axis limits
plt.legend(
    translated_labels, bbox_to_anchor=[1.22, 0.7], loc="center"
)  # Use translated labels for legend
plt.ylim(0, 20)

# Show the plot
plt.show()

## Flexibility needs: only supply; residual demand; aggregated as one node

In [ ]:
# assert 0

### Daily

In [ ]:
n = networks[2020]
s, d = supply_demand(
    n, interconnectors=True, merge_dist_grid=True, drop_dist_grid=False
)

In [ ]:
# simple testing
n = networks[2045]
electricity_supply, electricity_demand = supply_demand(
    n, interconnectors=True, merge_dist_grid=True, drop_dist_grid=False
)

non_vre_gens = electricity_supply[~electricity_supply.index.isin(vre_gens)].index
vre_gen = electricity_supply.loc[vre_gens].sum()
load = electricity_demand.loc[load_carriers].sum()
demand = electricity_demand.sum()

residual_load = (
    load - vre_gen - electricity_supply.loc["AC"] + electricity_demand.loc["AC"]
)

# Compute daily average of residual load
daily_average_residual = residual_load.resample("D").mean()

# Expand daily averages to hourly resolution for comparison
daily_average_hourly = daily_average_residual.reindex(
    residual_load.index, method="ffill"
)

# Compute the difference between residual load and daily average (only the positive part)
hourly_flexibility_needs = (residual_load - daily_average_hourly).clip(lower=0)

# Calculate daily flexibility needs
daily_flexibility_needs = hourly_flexibility_needs.resample("D").sum()

# Calculate total annual flexibility needs
total_annual_flexibility = hourly_flexibility_needs.sum()

print(f"Total annual flexibility needs: {round(total_annual_flexibility / 1e6, 2)} TWh")

In [ ]:
years = np.arange(2020, 2050, 5)
result_single = pd.DataFrame(index=flex_techs.keys(), columns=years)
result_cumulative = pd.DataFrame(index=flex_techs.keys(), columns=years)

for year in years:
    n = networks[year]

    for cumulative in [False, True]:
        if cumulative:
            electricity_supply, electricity_demand = supply_demand(
                n, interconnectors=True, merge_dist_grid=True, drop_dist_grid=False
            )
        else:
            electricity_supply, electricity_demand = supply_demand(
                n, interconnectors=False, merge_dist_grid=True, drop_dist_grid=False
            )

        load_carriers = [
            "electricity",
            "agriculture electricity",
            "industry electricity",
        ]
        vre_gens = [
            "onwind",
            "offwind-ac",
            "offwind-dc",
            "solar",
            "solar-hsat",
            "solar rooftop",
            "ror",
        ]
        non_vre_gens = electricity_supply[
            ~electricity_supply.index.isin(vre_gens)
        ].index

        vre_gen = electricity_supply.loc[vre_gens].sum()
        load = electricity_demand.loc[load_carriers].sum()
        demand = electricity_demand.sum()

        residual_load = demand - vre_gen

        for flex_tech in flex_techs.keys():
            # 1. Calculate initial residual load
            if flex_tech == "no":
                sub = 0
            elif all(
                item in electricity_supply.index for item in flex_techs[flex_tech]
            ):
                if len(flex_techs[flex_tech]) > 1:
                    sub = electricity_supply.loc[flex_techs[flex_tech]].sum()
                else:
                    sub = electricity_supply.loc[flex_techs[flex_tech][0]]
            else:
                print(f"Flex tech {flex_tech} not in electricity supply")
                continue

            if cumulative:
                residual_load = residual_load - sub
            else:
                residual_load = demand - vre_gen - sub

            # 2. Compute daily average of residual load
            daily_average_residual = residual_load.resample("D").mean()

            # Expand daily averages to hourly resolution for comparison
            daily_average_hourly = daily_average_residual.reindex(
                residual_load.index, method="ffill"
            )

            # 3. Compute the difference between residual load and daily average (only the positive part)
            hourly_flexibility_needs = (residual_load - daily_average_hourly).clip(
                lower=0
            )

            # Calculate daily flexibility needs
            daily_flexibility_needs = hourly_flexibility_needs.resample("D").sum()

            # 4. Calculate total annual flexibility needs
            total_annual_flexibility = hourly_flexibility_needs.sum()

            if cumulative:
                result_cumulative.loc[flex_tech, year] = round(
                    total_annual_flexibility / 1e6, 2
                )
            else:
                result_single.loc[flex_tech, year] = round(
                    total_annual_flexibility / 1e6, 2
                )

In [ ]:
# if interconnectors (AC, DC) are included the single tech contribution makes no sense: drop those before calculating
df_all_single = pd.DataFrame()

for year in np.arange(2020, 2050, 5):
    df = result_single[year]
    no_total_i = df.drop("no").index
    # calc flexibility contribution
    df[no_total_i] = -(result_single[year] - result_single[year].loc["no"])[no_total_i]
    df = df[df > 1e-3]
    df_all_single = pd.concat([df_all_single, df], axis=1)

df_all_single

In [ ]:
result_cumulative[2045]

In [ ]:
# if interconnectors are off for the single tech contribution and on for the cumulative contribution, the values are rather similar
df_all_cum = pd.DataFrame()

for year in np.arange(2020, 2050, 5):
    df = result_cumulative[year].dropna().diff().abs()
    df = df[df > 1e-3]
    df_all_cum = pd.concat([df_all_cum, df], axis=1)

df_all_cum  # here in TWh total (plot is in GWh/d)

In [ ]:
df = df_all_cum * 1e3 / 365  # df_all_single.drop(["no"], errors="ignore")
tech_colors["battery"] = "fuchsia"

# Set plot properties
fig, ax = plt.subplots(figsize=(8, 5))  # Increase figsize

# Extract colors for the bar plot
colors = [tech_colors[tech] for tech in df.index]

# Create the bar plot
df.T.plot(kind="bar", stacked=True, color=colors, ax=ax)

# Add title and labels
ax.set_title("Tägliche Flexibilitätsbedarfe", fontsize=16)
ax.set_ylabel("GWh / Tag", fontsize=14)

# Move the legend outside the plot
ax.legend(
    loc="upper left",
    bbox_to_anchor=(1, 1),  # Position the legend outside
    title="Flexibilitätstechnologien",
)

# Tighten the layout to fit everything
plt.tight_layout()

# Display the plot
plt.show()

In [ ]:
result_cumulative.T.plot(kind="bar", figsize=(20, 6))

In [ ]:
result_single

### Weekly

In [ ]:
# simple testing

n = networks[2045]
electricity_supply, electricity_demand = supply_demand(
    n, interconnectors=True, merge_dist_grid=True, drop_dist_grid=False
)

load_carriers = ["electricity", "agriculture electricity", "industry electricity"]
vre_gens = [
    "onwind",
    "offwind-ac",
    "offwind-dc",
    "solar",
    "solar-hsat",
    "solar rooftop",
    "ror",
]
non_vre_gens = electricity_supply[~electricity_supply.index.isin(vre_gens)].index

vre_gen = electricity_supply.loc[vre_gens].sum()
load = electricity_demand.loc[load_carriers].sum()
demand = electricity_demand.sum()

residual_load = (
    demand - vre_gen - electricity_supply.loc["AC"]
)  # - electricity_supply.loc["urban central H2 CHP"]

# Compute daily average of residual load
daily_average_residual = residual_load.resample("D").mean()

# Expand daily averages to hourly resolution for comparison
daily_average_hourly = daily_average_residual.reindex(
    residual_load.index, method="ffill"
)

# Compute weekly average of daily averages
weekly_average_residual = daily_average_residual.resample("W").mean()
weekly_average_hourly = weekly_average_residual.reindex(
    residual_load.index, method="ffill"
)

# Compute the difference between daily average and weekly average (only the positive part)
weekly_flexibility_needs = (daily_average_hourly - weekly_average_hourly).clip(lower=0)

# Calculate weekly flexibility needs
weekly_flexibility_needs = weekly_flexibility_needs.resample("W").sum()

# Calculate total weekly flexibility needs for one year (TWh/a)
total_annual_flexibility = weekly_flexibility_needs.sum()

print(
    f"Total weekly flexibility needs for one year (TWh/a): {round(total_annual_flexibility / 1e6, 2)} TWh"
)

In [ ]:
# 30s

years = np.arange(2020, 2050, 5)
result_single = pd.DataFrame(index=flex_techs.keys(), columns=years)
result_cumulative = pd.DataFrame(index=flex_techs.keys(), columns=years)

for year in years:
    n = networks[year]

    for cumulative in [False, True]:
        if cumulative:
            electricity_supply, electricity_demand = supply_demand(
                n, interconnectors=True, merge_dist_grid=True, drop_dist_grid=False
            )
        else:
            electricity_supply, electricity_demand = supply_demand(
                n, interconnectors=False, merge_dist_grid=True, drop_dist_grid=False
            )

        load_carriers = [
            "electricity",
            "agriculture electricity",
            "industry electricity",
        ]
        vre_gens = [
            "onwind",
            "offwind-ac",
            "offwind-dc",
            "solar",
            "solar-hsat",
            "solar rooftop",
            "ror",
        ]
        non_vre_gens = electricity_supply[
            ~electricity_supply.index.isin(vre_gens)
        ].index

        vre_gen = electricity_supply.loc[vre_gens].sum()
        load = electricity_demand.loc[load_carriers].sum()
        demand = electricity_demand.sum()

        residual_load = demand - vre_gen

        for flex_tech in flex_techs.keys():
            # 1. Calculate initial residual load
            if flex_tech == "no":
                sub = 0
            elif all(
                item in electricity_supply.index for item in flex_techs[flex_tech]
            ):
                if len(flex_techs[flex_tech]) > 1:
                    sub = electricity_supply.loc[flex_techs[flex_tech]].sum()
                else:
                    sub = electricity_supply.loc[flex_techs[flex_tech][0]]
            else:
                print(f"Flex tech {flex_tech} not in electricity supply")
                continue

            if cumulative:
                residual_load = residual_load - sub
            else:
                residual_load = demand - vre_gen - sub

            # Compute daily average of residual load
            daily_average_residual = residual_load.resample("D").mean()

            # Expand daily averages to hourly resolution for comparison
            daily_average_hourly = daily_average_residual.reindex(
                residual_load.index, method="ffill"
            )

            # Compute weekly average of daily averages
            weekly_average_residual = daily_average_residual.resample("W").mean()
            weekly_average_hourly = weekly_average_residual.reindex(
                residual_load.index, method="ffill"
            )

            # Compute the difference between daily average amy weekly average (only the positive part)
            weekly_flexibility_needs = (
                daily_average_hourly - weekly_average_hourly
            ).clip(lower=0)

            # Calculate weekly flexibility needs
            weekly_flexibility_needs = weekly_flexibility_needs.resample("W").sum()

            # Calculate total weekly flexibility needs for one year (TWh/a)
            total_annual_flexibility = weekly_flexibility_needs.sum()

            if cumulative:
                result_cumulative.loc[flex_tech, year] = round(
                    total_annual_flexibility / 1e6, 2
                )
            else:
                result_single.loc[flex_tech, year] = round(
                    total_annual_flexibility / 1e6, 2
                )

In [ ]:
df_all_single = pd.DataFrame()

for year in np.arange(2020, 2050, 5):
    df = result_single[year].dropna().diff().abs()
    df = df[df > 0.5]
    df_all_single = pd.concat([df_all_single, df], axis=1)

df_all_single

In [ ]:
df_all_cum = pd.DataFrame()

for year in np.arange(2020, 2050, 5):
    df = result_cumulative[year].dropna().diff().abs()
    df = df[df > 0.5]
    df_all_cum = pd.concat([df_all_cum, df], axis=1)

df_all_cum

In [ ]:
df = df_all_cum.T / 52

# Set plot properties
fig, ax = plt.subplots(figsize=(8, 5))  # Increase figsize

# Extract colors for the bar plot
colors = [tech_colors[tech] for tech in df_all_cum.index]

# Create the bar plot
df.plot(kind="bar", stacked=True, color=colors, ax=ax)

# Add title and labels
ax.set_title("Wöchentliche Flexibilitätsbedarfe", fontsize=16)
ax.set_ylabel("TWh / Woche", fontsize=14)
# ax.set_xlabel("Year", fontsize=14)

# Move the legend outside the plot
ax.legend(
    loc="upper left",
    bbox_to_anchor=(1, 1),  # Position the legend outside
    title="Flexibilitätstechnologien",
)

# Tighten the layout to fit everything
plt.tight_layout()

# Display the plot
plt.show()

### Annual

In [ ]:
# simple testing
# annual flexibility needs are determined as the cumulated difference between the monthly averages and the mean residual load across the entire year

n = networks[2020]
electricity_supply, electricity_demand = supply_demand(
    n, interconnectors=True, merge_dist_grid=True, drop_dist_grid=False
)

load_carriers = ["electricity", "agriculture electricity", "industry electricity"]
vre_gens = [
    "onwind",
    "offwind-ac",
    "offwind-dc",
    "solar",
    "solar-hsat",
    "solar rooftop",
    "ror",
]
non_vre_gens = electricity_supply[~electricity_supply.index.isin(vre_gens)].index

vre_gen = electricity_supply.loc[vre_gens].sum()
load = electricity_demand.loc[load_carriers].sum()
demand = electricity_demand.sum()

residual_load = demand - vre_gen  # - electricity_supply.loc["PHS"]

# Compute daily average of residual load
daily_average_residual = residual_load.resample("D").mean()

# Expand daily averages to hourly resolution for comparison
daily_average_hourly = daily_average_residual.reindex(
    residual_load.index, method="bfill"
)

# Compute weekly average of daily averages
weekly_average_residual = daily_average_residual.resample("W").mean()
weekly_average_hourly = weekly_average_residual.reindex(
    residual_load.index, method="bfill"
)

# Compute monthly average of weekly averages
monthly_average_residual = weekly_average_residual.resample("ME").mean()
monthly_average_hourly = monthly_average_residual.reindex(
    residual_load.index, method="bfill"
)

# Compute yearly average of monthly averages
yearly_average_residual = monthly_average_residual.resample("YE").mean()
yearly_average_hourly = yearly_average_residual.reindex(
    residual_load.index, method="bfill"
)

# Compute the difference between monthly average and yearly (only the positive part)
yearly_flexibility_needs = (monthly_average_hourly - yearly_average_hourly).clip(
    lower=0
)

# Calculate weekly flexibility needs
yearly_flexibility_needs = yearly_flexibility_needs.resample("D").sum()

# Calculate total weekly flexibility needs for one year (TWh/a)
total_annual_flexibility = yearly_flexibility_needs.sum()

print(
    f"Total total weekly flexibility needs for one year (TWh/a): {round(total_annual_flexibility / 1e6, 2)} TWh"
)

In [ ]:
years = np.arange(2020, 2050, 5)
result_single = pd.DataFrame(index=flex_techs.keys(), columns=years)
result_cumulative = pd.DataFrame(index=flex_techs.keys(), columns=years)

for year in years:
    n = networks[year]

    for cumulative in [False, True]:
        if cumulative:
            electricity_supply, electricity_demand = supply_demand(
                n, interconnectors=True, merge_dist_grid=True, drop_dist_grid=False
            )
        else:
            electricity_supply, electricity_demand = supply_demand(
                n, interconnectors=False, merge_dist_grid=True, drop_dist_grid=False
            )

        load_carriers = [
            "electricity",
            "agriculture electricity",
            "industry electricity",
        ]
        vre_gens = [
            "onwind",
            "offwind-ac",
            "offwind-dc",
            "solar",
            "solar-hsat",
            "solar rooftop",
            "ror",
        ]
        non_vre_gens = electricity_supply[
            ~electricity_supply.index.isin(vre_gens)
        ].index

        vre_gen = electricity_supply.loc[vre_gens].sum()
        load = electricity_demand.loc[load_carriers].sum()
        demand = electricity_demand.sum()

        residual_load = demand - vre_gen

        for flex_tech in flex_techs.keys():
            # 1. Calculate initial residual load
            if flex_tech == "no":
                sub = 0
            elif all(
                item in electricity_supply.index for item in flex_techs[flex_tech]
            ):
                if len(flex_techs[flex_tech]) > 1:
                    sub = electricity_supply.loc[flex_techs[flex_tech]].sum()
                else:
                    sub = electricity_supply.loc[flex_techs[flex_tech][0]]
            else:
                print(f"Flex tech {flex_tech} not in electricity supply")
                continue

            if cumulative:
                residual_load = residual_load - sub
            else:
                residual_load = demand - vre_gen - sub

            # Compute daily average of residual load
            daily_average_residual = residual_load.resample("D").mean()

            # Expand daily averages to hourly resolution for comparison
            daily_average_hourly = daily_average_residual.reindex(
                residual_load.index, method="bfill"
            )

            # Compute weekly average of daily averages
            weekly_average_residual = daily_average_residual.resample("W").mean()
            weekly_average_hourly = weekly_average_residual.reindex(
                residual_load.index, method="bfill"
            )

            # Compute monthly average of weekly averages
            monthly_average_residual = weekly_average_residual.resample("ME").mean()
            monthly_average_hourly = monthly_average_residual.reindex(
                residual_load.index, method="bfill"
            )

            # Compute yearly average of monthly averages
            yearly_average_residual = monthly_average_residual.resample("YE").mean()
            yearly_average_hourly = yearly_average_residual.reindex(
                residual_load.index, method="bfill"
            )

            # Compute the difference between monthly average and yearly (only the positive part)
            yearly_flexibility_needs = (
                monthly_average_hourly - yearly_average_hourly
            ).clip(lower=0)

            # Calculate weekly flexibility needs
            yearly_flexibility_needs = yearly_flexibility_needs.resample("D").sum()

            # Calculate total weekly flexibility needs for one year (TWh/a)
            total_annual_flexibility = yearly_flexibility_needs.sum()

            if cumulative:
                result_cumulative.loc[flex_tech, year] = round(
                    total_annual_flexibility / 1e6, 2
                )
            else:
                result_single.loc[flex_tech, year] = round(
                    total_annual_flexibility / 1e6, 2
                )

In [ ]:
df_all_single = pd.DataFrame()

for year in np.arange(2020, 2050, 5):
    df = result_single[year].dropna().diff().abs()
    df = df[df > 0.5]
    df_all_single = pd.concat([df_all_single, df], axis=1)

df_all_single

In [ ]:
df_all_cum = pd.DataFrame()

for year in np.arange(2020, 2050, 5):
    df = result_cumulative[year].dropna().diff().abs()
    df = df[df > 0.5]
    df_all_cum = pd.concat([df_all_cum, df], axis=1)

df_all_cum

In [ ]:
# Set plot properties

df = df_all_single
fig, ax = plt.subplots(figsize=(8, 5))  # Increase figsize

# Extract colors for the bar plot
colors = [tech_colors[tech] for tech in df.index]

# Create the bar plot
df.T.plot(kind="bar", stacked=True, color=colors, ax=ax)

# Add title and labels
ax.set_title("Annual flexibility needs", fontsize=16)
ax.set_ylabel("TWh", fontsize=14)
ax.set_xlabel("Year", fontsize=14)

# Move the legend outside the plot
ax.legend(
    loc="upper left",
    bbox_to_anchor=(1, 1),  # Position the legend outside
    title="Flexibility Technologies",
)

# Tighten the layout to fit everything
plt.tight_layout()

# Display the plot
plt.show()

## Flexibility needs - supply and demand (aggregated as one node)

### Daily

In [ ]:
# simple testing
n = networks[2020]
electricity_supply, electricity_demand = supply_demand(
    n,
    interconnectors=False,
    merge_dist_grid=True,
    drop_dist_grid=False,
    add_diff_as_import=True,
)

non_vre_gens = electricity_supply[~electricity_supply.index.isin(vre_gens)].index
vre_gen = electricity_supply.loc[vre_gens].sum()
load = electricity_demand.loc[load_carriers].sum()

residual_load = load - vre_gen  # + electricity_demand.loc["export"]

# Compute daily average of residual load
daily_average_residual = residual_load.resample("D").mean()

# Expand daily averages to hourly resolution for comparison
daily_average_hourly = daily_average_residual.reindex(
    residual_load.index, method="ffill"
)

# Compute the difference between residual load and daily average (only the positive part)
hourly_flexibility_needs = (residual_load - daily_average_hourly).clip(lower=0)

# Calculate daily flexibility needs
daily_flexibility_needs = hourly_flexibility_needs.resample("D").sum()

# Calculate total annual flexibility needs
total_annual_flexibility = hourly_flexibility_needs.sum()

print(f"Total annual flexibility needs: {round(total_annual_flexibility / 1e6, 2)} TWh")

In [ ]:
load_carriers = ["electricity", "agriculture electricity", "industry electricity"]
vre_gens = [
    "onwind",
    "offwind-ac",
    "offwind-dc",
    "solar",
    "solar-hsat",
    "solar rooftop",
    "ror",
]

years = np.arange(2020, 2050, 5)

result_single = pd.DataFrame(index=all_flex_techs, columns=years)
result_cumulative = pd.DataFrame(index=all_flex_techs, columns=years)

for year in years:
    n = networks[year]

    for cumulative in [True, False]:
        if cumulative:
            electricity_supply, electricity_demand = supply_demand(
                n,
                interconnectors=False,
                merge_dist_grid=True,
                drop_dist_grid=False,
                add_diff_as_import=True,
            )
        else:
            electricity_supply, electricity_demand = supply_demand(
                n,
                interconnectors=False,
                merge_dist_grid=True,
                drop_dist_grid=False,
                add_diff_as_import=True,
            )

        non_vre_gens = electricity_supply[
            ~electricity_supply.index.isin(vre_gens)
        ].index

        vre_gen = electricity_supply.loc[vre_gens].sum()
        load = electricity_demand.loc[load_carriers].sum()

        residual_load = load - vre_gen

        for flex_tech in all_flex_techs:
            # 1. Calculate initial residual load
            if flex_tech == "no":
                sub = 0
            elif flex_tech in flex_techs_supply.keys():
                count = sum(
                    item in electricity_supply.index
                    for item in flex_techs_supply[flex_tech]
                )  # how many of the techs are present
                techs = [
                    item
                    for item in flex_techs_supply[flex_tech]
                    if item in electricity_supply.index
                ]  # which techs are present
                if count > 1:
                    sub = electricity_supply.loc[techs].sum()
                elif count == 1:
                    sub = electricity_supply.loc[techs[0]]
                else:
                    print(
                        f"Flex tech {flex_tech} not in electricity supply for year {year}"
                    )
                    continue
            elif flex_tech in flex_techs_demand.keys():
                count = sum(
                    item in electricity_demand.index
                    for item in flex_techs_demand[flex_tech]
                )
                techs = [
                    item
                    for item in flex_techs_demand[flex_tech]
                    if item in electricity_demand.index
                ]
                if count > 1:
                    sub = electricity_demand.loc[techs].sum() * -1
                elif count == 1:
                    sub = electricity_demand.loc[techs[0]] * -1
                else:
                    print(
                        f"Flex tech {flex_tech} not in electricity demand for year {year}"
                    )
                    continue
            else:
                print(f"Flex tech {flex_tech} not in any flex tech list")
                continue

            if cumulative:
                residual_load = residual_load - sub
            else:
                residual_load = load - vre_gen - sub

            # 2. Compute daily average of residual load
            daily_average_residual = residual_load.resample("D").mean()

            # Expand daily averages to hourly resolution for comparison
            daily_average_hourly = daily_average_residual.reindex(
                residual_load.index, method="ffill"
            )

            # 3. Compute the difference between residual load and daily average (only the positive part)
            hourly_flexibility_needs = (residual_load - daily_average_hourly).clip(
                lower=0
            )

            # Calculate daily flexibility needs
            daily_flexibility_needs = hourly_flexibility_needs.resample("D").sum()

            # 4. Calculate total annual flexibility needs
            total_annual_flexibility = hourly_flexibility_needs.sum()

            if cumulative:
                result_cumulative.loc[flex_tech, year] = round(
                    total_annual_flexibility / 1e6, 2
                )
            else:
                result_single.loc[flex_tech, year] = round(
                    total_annual_flexibility / 1e6, 2
                )

#### Single

In [ ]:
df_all_single_d = pd.DataFrame()

for year in np.arange(2020, 2050, 5):
    df = result_single[year]
    no_total_i = df.drop("no").index
    # calc flexibility contribution
    df[no_total_i] = -(result_single[year] - result_single[year].loc["no"])[no_total_i]
    df_all_single_d = pd.concat([df_all_single_d, df], axis=1)

df_all_single_d

In [ ]:
df = df_all_single_d.drop(["no"], errors="ignore")

# Set plot properties
fig, ax = plt.subplots(figsize=(8, 5))  # Increase figsize

# Extract colors for the bar plot
colors = [tech_colors[tech] for tech in df.index]

# Create the bar plot
df.T.plot(kind="bar", stacked=True, color=colors, ax=ax)

# Add title and labels
ax.set_title("Tägliche Flexibilitätsbedarfe", fontsize=16)
ax.set_ylabel("GWh / Tag", fontsize=14)

# Move the legend outside the plot
ax.legend(
    loc="upper left",
    bbox_to_anchor=(1, 1),  # Position the legend outside
    title="Flexibilitätstechnologien",
)

# Tighten the layout to fit everything
plt.tight_layout()

# Display the plot
plt.show()

In [ ]:
# nice plot
df_single_overall_light_d = df_all_single_d.copy()
df_single_overall_light_d.loc["Import/Export"] = df_single_overall_light_d.loc[
    ["import", "export"]
].sum()
df_single_overall_light_d.loc["PHS"] = df_single_overall_light_d.loc[
    ["PHS charging", "PHS discharging"]
].sum()
df_single_overall_light_d = df_single_overall_light_d.drop(
    [
        "import",
        "export",
        "interconnectors supply",
        "interconnectors demand",
        "PHS charging",
        "PHS discharging",
    ]
)

# drop rows which have less than 1 % of the total flexibility needs in all years
shares = df_single_overall_light_d / df_single_overall_light_d.drop("no").sum()
techs = shares[shares > 0.01].dropna(how="all").index
df_single_overall_light_d = df_single_overall_light_d.loc[techs]
sorted_techs = df_single_overall_light_d.sum(axis=1).sort_values(ascending=False).index
df_single_overall_light_d = df_single_overall_light_d.loc[sorted_techs]
df_single_overall_light_d.loc["Sonstige"] = (
    df_single_overall_light_d.loc["no"] - df_single_overall_light_d.drop("no").sum()
)
df_single_overall_light_d = df_single_overall_light_d.drop("no")

In [ ]:
df_single_overall_light_d * 1e3 / 365

In [ ]:
df = df_single_overall_light_d / 1e3 * 365

# Set plot properties
fig, ax = plt.subplots(figsize=(8, 5))
# Extract colors for the bar plot
colors = [tech_colors[tech] for tech in df.index]

# Create the bar plot
df.T.plot(kind="bar", stacked=True, color=colors, ax=ax)

# Add title and labels
# ax.set_title("Tägliche Flexibilitätsbedarfe", fontsize=16)
ax.set_ylabel("GWh / Tag", fontsize=14)

# Move the legend outside the plot
ax.legend(
    labels=[carriers_in_german.get(l, l) for l in df.index],
    loc="upper left",
    bbox_to_anchor=(1, 1),  # Position the legend outside
    title="Flexibilitätstechnologien",
)

# Tighten the layout to fit everything
plt.tight_layout()

plt.savefig(PLOT_DIR + "/daily_flex_single_nice")

In [ ]:
df = df_single_overall_light_d

# Set plot properties
fig, ax = plt.subplots(figsize=(7, 7))
# Extract colors for the bar plot
colors = [tech_colors[tech] for tech in df.index]

# Create the bar plot
df.T.plot(kind="bar", stacked=True, color=colors, ax=ax)

# Add title and labels
# ax.set_title("Tägliche Flexibilitätsbedarfe", fontsize=16)
ax.set_ylabel("TWh / a", fontsize=14)

# Move the legend outside the plot
ax.legend(
    labels=[carriers_in_german.get(l, l) for l in df.index],
    loc="upper left",
    # bbox_to_anchor=(1, 1),  # Position the legend outside
    bbox_to_anchor=(-0.025, -0.15),
    ncol=3,
    title="Flexibilitätstechnologien",
)

# Tighten the layout to fit everything
plt.tight_layout()

plt.savefig(PLOT_DIR + "/daily_flex_single_nice_new")

In [ ]:
df = df_single_overall_light_d

# Set plot properties
fig, ax = plt.subplots(figsize=(7, 7))
# Extract colors for the bar plot
colors = [tech_colors[tech] for tech in df.index]

# Create the bar plot
df.T.plot(kind="bar", stacked=True, color=colors, ax=ax)

# Add title and labels
# ax.set_title("Tägliche Flexibilitätsbedarfe", fontsize=16)
ax.set_ylabel("TWh / a", fontsize=14)

# Move the legend outside the plot
ax.legend(
    loc="upper left",
    # bbox_to_anchor=(1, 1),  # Position the legend outside
    bbox_to_anchor=(-0.025, -0.15),
    ncol=3,
    title="flexibility technologies",
)

# Tighten the layout to fit everything
plt.tight_layout()

plt.savefig(PLOT_DIR + "/daily_flex_single_nice_new_english")

In [ ]:
df_single_overall_light_d

#### Cumulative

In [ ]:
df_all_cum_d = pd.DataFrame()

for year in np.arange(2020, 2050, 5):
    df = result_cumulative[year].dropna().diff() * -1
    df_all_cum_d = pd.concat([df_all_cum_d, df], axis=1)

df_all_cum_d

In [ ]:
df = df_all_cum_d.drop(["no"], errors="ignore")

# Set plot properties
fig, ax = plt.subplots(figsize=(8, 5))  # Increase figsize

# Extract colors for the bar plot
colors = [tech_colors[tech] for tech in df.index]

# Create the bar plot
df.T.plot(kind="bar", stacked=True, color=colors, ax=ax)

# Add title and labels
ax.set_title("Tägliche Flexibilitätsbedarfe", fontsize=16)
ax.set_ylabel("GWh / Tag", fontsize=14)

# Move the legend outside the plot
ax.legend(
    loc="upper left",
    bbox_to_anchor=(1, 1),  # Position the legend outside
    title="Flexibilitätstechnologien",
)

# Tighten the layout to fit everything
plt.tight_layout()

# Display the plot
plt.show()

### Weekly

In [ ]:
years = np.arange(2020, 2050, 5)

result_single = pd.DataFrame(index=all_flex_techs, columns=years)
result_cumulative = pd.DataFrame(index=all_flex_techs, columns=years)

for year in years:
    n = networks[year]

    for cumulative in [True, False]:
        if cumulative:
            electricity_supply, electricity_demand = supply_demand(
                n,
                interconnectors=False,
                merge_dist_grid=True,
                drop_dist_grid=False,
                add_diff_as_import=True,
            )
        else:
            electricity_supply, electricity_demand = supply_demand(
                n,
                interconnectors=False,
                merge_dist_grid=True,
                drop_dist_grid=False,
                add_diff_as_import=True,
            )

        non_vre_gens = electricity_supply[
            ~electricity_supply.index.isin(vre_gens)
        ].index

        vre_gen = electricity_supply.loc[vre_gens].sum()
        load = electricity_demand.loc[load_carriers].sum()

        residual_load = load - vre_gen

        for flex_tech in all_flex_techs:
            # 1. Calculate initial residual load
            if flex_tech == "no":
                sub = 0
            elif flex_tech in flex_techs_supply.keys():
                count = sum(
                    item in electricity_supply.index
                    for item in flex_techs_supply[flex_tech]
                )  # how many of the techs are present
                techs = [
                    item
                    for item in flex_techs_supply[flex_tech]
                    if item in electricity_supply.index
                ]  # which techs are present
                if count > 1:
                    sub = electricity_supply.loc[techs].sum()
                elif count == 1:
                    sub = electricity_supply.loc[techs[0]]
                else:
                    print(
                        f"Flex tech {flex_tech} not in electricity supply for year {year}"
                    )
                    continue
            elif flex_tech in flex_techs_demand.keys():
                count = sum(
                    item in electricity_demand.index
                    for item in flex_techs_demand[flex_tech]
                )
                techs = [
                    item
                    for item in flex_techs_demand[flex_tech]
                    if item in electricity_demand.index
                ]
                if count > 1:
                    sub = electricity_demand.loc[techs].sum() * -1
                elif count == 1:
                    sub = electricity_demand.loc[techs[0]] * -1
                else:
                    print(
                        f"Flex tech {flex_tech} not in electricity demand for year {year}"
                    )
                    continue
            else:
                print(f"Flex tech {flex_tech} not in any flex tech list")
                continue

            if cumulative:
                residual_load = residual_load - sub
            else:
                residual_load = load - vre_gen - sub

            # Compute daily average of residual load
            daily_average_residual = residual_load.resample("D").mean()

            # Expand daily averages to hourly resolution for comparison
            daily_average_hourly = daily_average_residual.reindex(
                residual_load.index, method="ffill"
            )

            # Compute weekly average of daily averages
            weekly_average_residual = daily_average_residual.resample("W").mean()
            weekly_average_hourly = weekly_average_residual.reindex(
                residual_load.index, method="ffill"
            )

            # Compute the difference between daily average amy weekly average (only the positive part)
            weekly_flexibility_needs = (
                daily_average_hourly - weekly_average_hourly
            ).clip(lower=0)

            # Calculate weekly flexibility needs
            weekly_flexibility_needs = weekly_flexibility_needs.resample("W").sum()

            # Calculate total weekly flexibility needs for one year (TWh/a)
            total_annual_flexibility = weekly_flexibility_needs.sum()

            if cumulative:
                result_cumulative.loc[flex_tech, year] = round(
                    total_annual_flexibility / 1e6, 2
                )
            else:
                result_single.loc[flex_tech, year] = round(
                    total_annual_flexibility / 1e6, 2
                )

#### Single

In [ ]:
df_all_single_w = pd.DataFrame()

for year in np.arange(2020, 2050, 5):
    df = result_single[year]
    no_total_i = df.drop("no").index
    # calc flexibility contribution
    df[no_total_i] = -(result_single[year] - result_single[year].loc["no"])[no_total_i]
    df_all_single_w = pd.concat([df_all_single_w, df], axis=1)

df_all_single_w

In [ ]:
df = df_all_single_w.drop(["no"], errors="ignore")

# Set plot properties
fig, ax = plt.subplots(figsize=(8, 5))  # Increase figsize

# Extract colors for the bar plot
colors = [tech_colors[tech] for tech in df.index]

# Create the bar plot
df.T.plot(kind="bar", stacked=True, color=colors, ax=ax)

# Add title and labels
ax.set_title("Wöchentliche Flexibilitätsbedarfe", fontsize=16)
ax.set_ylabel("TWh/a", fontsize=14)

# Move the legend outside the plot
ax.legend(
    loc="upper left",
    bbox_to_anchor=(1, 1),  # Position the legend outside
    title="Flexibilitätstechnologien",
)

# Tighten the layout to fit everything
plt.tight_layout()

# Display the plot
plt.show()

In [ ]:
# nice plot
df_single_overall_light_w = df_all_single_w.copy()
df_single_overall_light_w.loc["Import/Export"] = df_single_overall_light_w.loc[
    ["import", "export"]
].sum()
df_single_overall_light_w.loc["PHS"] = df_single_overall_light_w.loc[
    ["PHS charging", "PHS discharging"]
].sum()
df_single_overall_light_w = df_single_overall_light_w.drop(
    [
        "import",
        "export",
        "interconnectors supply",
        "interconnectors demand",
        "PHS charging",
        "PHS discharging",
    ]
)

# drop rows which have less than 1 % of the total flexibility needs in all years
shares = df_single_overall_light_w / df_single_overall_light_w.drop("no").sum()
techs = shares[shares > 0.01].dropna(how="all").index
df_single_overall_light_w = df_single_overall_light_w.loc[techs]
sorted_techs = df_single_overall_light_w.sum(axis=1).sort_values(ascending=False).index
df_single_overall_light_w = df_single_overall_light_w.loc[sorted_techs]
df_single_overall_light_w.loc["Sonstige"] = (
    df_single_overall_light_w.loc["no"] - df_single_overall_light_w.drop("no").sum()
)
df_single_overall_light_w = df_single_overall_light_w.drop("no")

In [ ]:
df_single_overall_light_w / 52

In [ ]:
df = df_single_overall_light_w / 52

# Set plot properties
fig, ax = plt.subplots(figsize=(8, 5))
# Extract colors for the bar plot
colors = [tech_colors[tech] for tech in df.index]

# Create the bar plot
df.T.plot(kind="bar", stacked=True, color=colors, ax=ax)

# Add title and labels
# ax.set_title("Wöchentliche Flexibilitätsbedarfe", fontsize=16)
ax.set_ylabel("TWh / Woche", fontsize=14)

# Move the legend outside the plot
ax.legend(
    labels=[carriers_in_german.get(l, l) for l in df.index],
    loc="upper left",
    bbox_to_anchor=(1, 1),  # Position the legend outside
    title="Flexibilitätstechnologien",
)

# Tighten the layout to fit everything
plt.tight_layout()

plt.savefig(PLOT_DIR + "/weekly_flex_single_nice")

In [ ]:
df = df_single_overall_light_w

# Set plot properties
fig, ax = plt.subplots(figsize=(7, 7))
# Extract colors for the bar plot
colors = [tech_colors[tech] for tech in df.index]

# Create the bar plot
df.T.plot(kind="bar", stacked=True, color=colors, ax=ax)

# Add title and labels
# ax.set_title("Wöchentliche Flexibilitätsbedarfe", fontsize=16)
ax.set_ylabel("TWh / a", fontsize=14)

# Move the legend outside the plot
ax.legend(
    labels=[carriers_in_german.get(l, l) for l in df.index],
    loc="upper left",
    bbox_to_anchor=(-0.025, -0.15),
    ncol=3,
    title="Flexibilitätstechnologien",
)

# Tighten the layout to fit everything
plt.tight_layout()

plt.savefig(PLOT_DIR + "/weekly_flex_single_nice_new")

In [ ]:
# adjust tech_colors

# carrier names manual
tech_colors["battery discharger"] = n.carriers.color["battery discharger"]
tech_colors["urban central oil CHP"] = n.carriers.color["oil"]
tech_colors["Solar"] = n.carriers.color["solar"]
tech_colors["Electricity load"] = n.carriers.color["electricity"]
tech_colors["Electricity trade"] = n.carriers.color["AC"]
tech_colors["Offshore Wind"] = n.carriers.color["offwind-ac"]
tech_colors["urban decentral heat"] = n.carriers.color["urban central heat"]
tech_colors["H2 OCGT"] = "#3b4cc0"
tech_colors["H2 retrofit OCGT"] = "#9abbff"
tech_colors["urban central H2 CHP"] = "#c9d7f0"
tech_colors["urban central H2 retrofit CHP"] = "#edd1c2"
tech_colors["H2 Electrolysis"] = "#9b2d91"  # "#d93ccd"

### Ariadne plot

In [ ]:
# Set up the subplots (1 row, 2 columns)
fig, ax = plt.subplots(1, 2, figsize=(10, 5), sharey=True)

# Plot for daily flexibility needs (df_single_overall_light_d)
colors_d = [tech_colors[tech] for tech in df_single_overall_light_d.index]
df_single_overall_light_d.T.plot(
    kind="bar",
    stacked=True,
    color=colors_d,
    ax=ax[0],
    legend=False,  # Disable legend for the first plot
)
ax[0].set_ylabel("TWh/a", fontsize=14)
ax[0].set_title("Tägliche Flexibilitätsbedarfe [TWh/a]", fontsize=16)

# Plot for weekly flexibility needs (df_single_overall_light_w)
colors_w = [tech_colors[tech] for tech in df_single_overall_light_w.index]
df_single_overall_light_w.T.plot(
    kind="bar",
    stacked=True,
    color=colors_w,
    ax=ax[1],
    legend=False,  # Disable legend for the second plot
)
ax[1].set_title("Wöchentliche Flexibilitätsbedarfe [TWh/a]", fontsize=16)

# Combine the distinct labels from both plots
# Get the unique technologies from both DataFrames
all_technologies = set(df_single_overall_light_d.index).union(
    df_single_overall_light_w.index
)
# Sort to keep a consistent order
all_technologies = sorted(all_technologies)

# Create the legend handles and labels for all distinct technologies
handles, labels = [], []
for tech in all_technologies:
    handles.append(
        plt.Line2D(
            [0],
            [0],
            marker="s",
            color="w",
            markerfacecolor=tech_colors[tech],
            markersize=10,
        )
    )
    labels.append(carriers_in_german.get(tech, tech))

# Place the legend below both subplots
fig.legend(
    handles=handles,
    labels=labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0),  # Position below both plots
    ncol=5,
    title="Flexibilitätstechnologien",
)

# Tighten layout to ensure everything fits
plt.tight_layout()

# Save the combined plot
plt.savefig(PLOT_DIR + "/flexibility_needs_combined.png", bbox_inches="tight")

# save plot
plt.savefig(
    f"/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flexibility_needs_combined_{scenario}_{weather_year}.png",
    bbox_inches="tight",
)

# save df
df_single_overall_light_d.to_csv(
    f"/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_contribution_d_{scenario}_{weather_year}.csv"
)
df_single_overall_light_w.to_csv(
    f"/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_contribution_w_{scenario}_{weather_year}.csv"
)

# Show the plot
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

# Plot for daily flexibility needs (df_single_overall_light_d)
colors_d = [tech_colors[tech] for tech in df_single_overall_light_d.index]
df_single_overall_light_d.T.plot(
    kind="bar",
    stacked=True,
    color=colors_d,
    ax=ax,
    legend=False,  # Disable legend for the first plot
)
ax.set_ylabel("TWh/a", fontsize=14)
plt.xticks(rotation=0)
ax.set_title("Tägliche Flexibilitätsbedarfe [TWh/a]", fontsize=16)
german_legend_labels = [
    carriers_in_german.get(tech, tech) for tech in df_single_overall_light_d.index
]
ax.legend(
    german_legend_labels,
    loc="upper left",
    bbox_to_anchor=(1, 1),
    title="Flexibilitätstechnologien",
)
plt.savefig(PLOT_DIR + "/flexibility_needs_daily.pdf", bbox_inches="tight")

In [ ]:
# Set up the subplots (1 row, 2 columns)
fig, ax = plt.subplots(1, 2, figsize=(10, 5), sharey=True)

# Plot for daily flexibility needs (df_single_overall_light_d)
colors_d = [tech_colors[tech] for tech in df_single_overall_light_d.index]
df_single_overall_light_d.T.plot(
    kind="bar",
    stacked=True,
    color=colors_d,
    ax=ax[0],
    legend=False,  # Disable legend for the first plot
)
ax[0].set_ylabel("TWh / a", fontsize=14)
ax[0].set_title("Daily flexibility needs", fontsize=16)

# Plot for weekly flexibility needs (df_single_overall_light_w)
colors_w = [tech_colors[tech] for tech in df_single_overall_light_w.index]
df_single_overall_light_w.T.plot(
    kind="bar",
    stacked=True,
    color=colors_w,
    ax=ax[1],
    legend=False,  # Disable legend for the second plot
)
ax[1].set_title("Weekly flexibility needs", fontsize=16)

# Combine the distinct labels from both plots
# Get the unique technologies from both DataFrames
all_technologies = set(df_single_overall_light_d.index).union(
    df_single_overall_light_w.index
)
# Sort to keep a consistent order
all_technologies = sorted(all_technologies)

# Create the legend handles and labels for all distinct technologies
handles, labels = [], []
for tech in all_technologies:
    handles.append(
        plt.Line2D(
            [0],
            [0],
            marker="s",
            color="w",
            markerfacecolor=tech_colors[tech],
            markersize=10,
        )
    )
    labels.append(tech if tech != "Sonstige" else "other")


# Place the legend below both subplots
fig.legend(
    handles=handles,
    labels=labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0),  # Position below both plots
    ncol=5,
    title="flexibility technologies",
)

# Tighten layout to ensure everything fits
plt.tight_layout()

# Save the combined plot
plt.savefig(PLOT_DIR + "/flexibility_needs_combined_english", bbox_inches="tight")

# Show the plot
plt.show()

In [ ]:
df_single_overall_light_d

In [ ]:
df_single_overall_light_d.to_excel("df_single_overall_light_d.xlsx", index=True)

In [ ]:
df_single_overall_light_w

In [ ]:
df_single_overall_light_w.to_excel("df_single_overall_light_w.xlsx", index=True)

In [ ]:
fc_cp_2019_d = pd.read_csv(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_contribution_d_CurrentPolicies_2019.csv",
    index_col=0,
)
fc_cp_2019_w = pd.read_csv(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_contribution_w_CurrentPolicies_2019.csv",
    index_col=0,
)
fc_mix_2019_d = pd.read_csv(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_contribution_d_KN2045_Bal_v4_2019.csv",
    index_col=0,
)
fc_mix_2019_w = pd.read_csv(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_contribution_w_KN2045_Bal_v4_2019.csv",
    index_col=0,
)
fc_elec_2019_d = pd.read_csv(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_contribution_d_KN2045_Elec_v4_2019.csv",
    index_col=0,
)
fc_elec_2019_w = pd.read_csv(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_contribution_w_KN2045_Elec_v4_2019.csv",
    index_col=0,
)
fc_h2_2019_d = pd.read_csv(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_contribution_d_KN2045_H2_v4_2019.csv",
    index_col=0,
)
fc_h2_2019_w = pd.read_csv(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_contribution_w_KN2045_H2_v4_2019.csv",
    index_col=0,
)
fc_mix_1996_d = pd.read_csv(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_contribution_d_Mix_1996.csv",
    index_col=0,
)
fc_mix_1996_w = pd.read_csv(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_contribution_w_Mix_1996.csv",
    index_col=0,
)
fc_mix_2010_d = pd.read_csv(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_contribution_d_Mix_2010.csv",
    index_col=0,
)
fc_mix_2010_w = pd.read_csv(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_contribution_w_Mix_2010.csv",
    index_col=0,
)
fc_mix_2013_d = pd.read_csv(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_contribution_d_Mix_2013.csv",
    index_col=0,
)
fc_mix_2013_w = pd.read_csv(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_contribution_w_Mix_2013.csv",
    index_col=0,
)

In [ ]:
fc_mix_2019_d

In [ ]:
fc_cp_2019_w

In [ ]:
import numpy as np

# Dictionary to store all dataframes
scenarios = {
    "Current Policies": {"daily": fc_cp_2019_d, "weekly": fc_cp_2019_w},
    "KN2045 Balanced": {"daily": fc_mix_2019_d, "weekly": fc_mix_2019_w},
    "KN2045 Electric": {"daily": fc_elec_2019_d, "weekly": fc_elec_2019_w},
    "KN2045 H2": {"daily": fc_h2_2019_d, "weekly": fc_h2_2019_w},
}

# Set up the subplots (1 row, 2 columns)
fig, ax = plt.subplots(1, 2, figsize=(16, 6), sharey=True)


# Function to plot multiple scenarios as grouped bars
def plot_scenarios(ax_subplot, data_dict, title):
    # Get all years from the first scenario
    years = data_dict[list(scenarios.keys())[0]].columns

    # Set up positions for grouped bars
    n_scenarios = len(scenarios)
    n_years = len(years)
    bar_width = 0.8 / n_scenarios
    x_positions = np.arange(n_years)

    # Get all unique technologies across all scenarios for this plot type
    all_technologies = set()
    for scenario_name in scenarios.keys():
        if scenario_name in data_dict and not data_dict[scenario_name].empty:
            all_technologies.update(data_dict[scenario_name].index)
    all_technologies = sorted(all_technologies)

    # Debug: print technologies found
    print(f"Technologies found for {title}: {all_technologies}")

    # Plot each scenario
    for i, (scenario_name, df) in enumerate(data_dict.items()):
        # Calculate x positions for this scenario
        x_pos = x_positions + (i - n_scenarios / 2 + 0.5) * bar_width

        # Prepare data for stacking
        bottom = np.zeros(len(years))

        # Plot each technology as part of the stack
        for tech in all_technologies:
            if tech in df.index:
                values = df.loc[tech].values
                # Handle NaN values by replacing with 0
                values = np.nan_to_num(values, nan=0.0)
            else:
                # If technology doesn't exist in this scenario, use zeros
                values = np.zeros(len(years))

            color = tech_colors.get(tech, "gray")
            # Only plot if there are non-zero values
            if np.any(values != 0):
                ax_subplot.bar(
                    x_pos,
                    values,
                    bar_width,
                    bottom=bottom,
                    color=color,
                    label=tech if i == 0 else "",
                )
            bottom += values

    # Customize the subplot
    ax_subplot.set_title(title, fontsize=16)
    ax_subplot.set_xticks(x_positions)
    ax_subplot.set_xticklabels(years, rotation=45)

    # Add scenario labels below x-axis
    scenario_labels = list(scenarios.keys())
    for i, scenario in enumerate(scenario_labels):
        x_pos = x_positions + (i - n_scenarios / 2 + 0.5) * bar_width
        for j, x in enumerate(x_pos):
            if j == len(x_pos) // 2:  # Only label in the middle
                ax_subplot.text(
                    x,
                    ax_subplot.get_ylim()[0]
                    - (ax_subplot.get_ylim()[1] - ax_subplot.get_ylim()[0]) * 0.1,
                    scenario,
                    ha="center",
                    va="top",
                    fontsize=8,
                    rotation=45,
                )


# Create data dictionaries for daily and weekly
daily_data = {name: info["daily"] for name, info in scenarios.items()}
weekly_data = {name: info["weekly"] for name, info in scenarios.items()}

# Plot daily flexibility needs
plot_scenarios(ax[0], daily_data, "Daily flexibility needs")
ax[0].set_ylabel("TWh / a", fontsize=14)

# Plot weekly flexibility needs
plot_scenarios(ax[1], weekly_data, "Weekly flexibility needs")

# Get all unique technologies for legend
all_technologies = set()
for scenario_info in scenarios.values():
    all_technologies.update(scenario_info["daily"].index)
    all_technologies.update(scenario_info["weekly"].index)
all_technologies = sorted(all_technologies)

# Create the legend handles and labels for all distinct technologies
handles, labels = [], []
for tech in all_technologies:
    if tech in tech_colors:
        handles.append(
            plt.Line2D(
                [0],
                [0],
                marker="s",
                color="w",
                markerfacecolor=tech_colors[tech],
                markersize=10,
            )
        )
        labels.append(tech if tech != "Sonstige" else "other")

# Place the legend below both subplots
fig.legend(
    handles=handles,
    labels=labels,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.05),
    ncol=5,
    title="flexibility technologies",
)

# Tighten layout to ensure everything fits
plt.tight_layout()

# Save the combined plot
plt.savefig(
    PLOT_DIR + "/flexibility_needs_scenarios_combined_english", bbox_inches="tight"
)

# Show the plot
plt.show()

In [ ]:
import numpy as np
import pandas as pd

# Define scenarios configuration with display names
scenario_config = {
    "Current Policies": {
        "file_suffix": "CurrentPolicies_2019",
        "display_name": "CP",  # Short name for plots
    },
    "KN2045 Balanced": {
        "file_suffix": "KN2045_Bal_v4_2019",
        "display_name": "Balanced",
    },
    "KN2045 Electric": {
        "file_suffix": "KN2045_Elec_v4_2019",
        "display_name": "Electric",
    },
    "KN2045 H2": {"file_suffix": "KN2045_H2_v4_2019", "display_name": "H2"},
}

# Base path for data files
base_path = (
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/"
)

# Load all scenarios automatically
scenarios = {}
for scenario_name, config in scenario_config.items():
    scenarios[scenario_name] = {
        "daily": pd.read_csv(
            f"{base_path}flex_contribution_d_{config['file_suffix']}.csv", index_col=0
        ),
        "weekly": pd.read_csv(
            f"{base_path}flex_contribution_w_{config['file_suffix']}.csv", index_col=0
        ),
        "display_name": config["display_name"],
    }

# Set up the subplots (1 row, 2 columns)
fig, ax = plt.subplots(1, 2, figsize=(16, 6), sharey=True)


# Function to plot multiple scenarios as grouped bars
def plot_scenarios(ax_subplot, data_dict, title):
    # Get all years from the first scenario
    years = data_dict[list(scenarios.keys())[0]].columns

    # Set up positions for grouped bars
    n_scenarios = len(scenarios)
    n_years = len(years)
    bar_width = 0.8 / n_scenarios
    x_positions = np.arange(n_years)

    # Get all unique technologies across all scenarios for this plot type
    all_technologies = set()
    for scenario_name in scenarios.keys():
        if scenario_name in data_dict and not data_dict[scenario_name].empty:
            all_technologies.update(data_dict[scenario_name].index)
    all_technologies = sorted(all_technologies)

    # Debug: print technologies found and check colors
    print(f"Technologies found for {title}: {all_technologies}")
    missing_colors = [tech for tech in all_technologies if tech not in tech_colors]
    if missing_colors:
        print(f"Technologies missing colors: {missing_colors}")

    # Plot each scenario
    for i, (scenario_name, df) in enumerate(data_dict.items()):
        # Calculate x positions for this scenario
        x_pos = x_positions + (i - n_scenarios / 2 + 0.5) * bar_width

        # Prepare data for stacking
        bottom = np.zeros(len(years))

        # Plot each technology as part of the stack
        for tech in all_technologies:
            if tech in df.index:
                values = df.loc[tech].values
                # Handle NaN values by replacing with 0
                values = np.nan_to_num(values, nan=0.0)
            else:
                # If technology doesn't exist in this scenario, use zeros
                values = np.zeros(len(years))

            color = tech_colors.get(tech, "gray")
            # Only plot if there are non-zero values
            if np.any(values != 0):
                ax_subplot.bar(
                    x_pos,
                    values,
                    bar_width,
                    bottom=bottom,
                    color=color,
                    label=tech if i == 0 else "",
                )
            bottom += values

    # Customize the subplot
    ax_subplot.set_title(title, fontsize=16)
    ax_subplot.set_xticks(x_positions)
    ax_subplot.set_xticklabels(years, rotation=45)

    # Add scenario labels below x-axis
    for i, scenario_name in enumerate(scenarios.keys()):
        display_name = scenarios[scenario_name]["display_name"]
        x_pos = x_positions + (i - n_scenarios / 2 + 0.5) * bar_width
        for j, x in enumerate(x_pos):
            if j == len(x_pos) // 2:  # Only label in the middle
                ax_subplot.text(
                    x,
                    ax_subplot.get_ylim()[0]
                    - (ax_subplot.get_ylim()[1] - ax_subplot.get_ylim()[0]) * 0.1,
                    display_name,
                    ha="center",
                    va="top",
                    fontsize=8,
                    rotation=45,
                )


# Create data dictionaries for daily and weekly
daily_data = {name: info["daily"] for name, info in scenarios.items()}
weekly_data = {name: info["weekly"] for name, info in scenarios.items()}

# Plot daily flexibility needs
plot_scenarios(ax[0], daily_data, "Daily flexibility needs")
ax[0].set_ylabel("TWh / a", fontsize=14)

# Plot weekly flexibility needs
plot_scenarios(ax[1], weekly_data, "Weekly flexibility needs")

# Get all unique technologies for legend
all_technologies_legend = set()
for scenario_info in scenarios.values():
    all_technologies_legend.update(scenario_info["daily"].index)
    all_technologies_legend.update(scenario_info["weekly"].index)
all_technologies_legend = sorted(all_technologies_legend)

# Create the legend handles and labels for all distinct technologies
handles, labels = [], []
for tech in all_technologies_legend:
    handles.append(
        plt.Line2D(
            [0],
            [0],
            marker="s",
            color="w",
            markerfacecolor=tech_colors.get(tech, "gray"),
            markersize=10,
        )
    )
    labels.append(tech if tech != "Sonstige" else "other")

# Place the legend below both subplots
fig.legend(
    handles=handles,
    labels=labels,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.05),
    ncol=5,
    title="flexibility technologies",
)

# Tighten layout to ensure everything fits
plt.tight_layout()

# Save the combined plot
plt.savefig(
    PLOT_DIR + "/flexibility_needs_scenarios_combined_english", bbox_inches="tight"
)

# Show the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Define scenarios configuration with display names
scenario_config = {
    "Current Policies": {
        "file_suffix": "CurrentPolicies_2019",
        "display_name": "CP",  # Short name for plots
    },
    "KN2045 Balanced": {
        "file_suffix": "KN2045_Bal_v4_2019",
        "display_name": "Balanced",
    },
    "KN2045 Electric": {
        "file_suffix": "KN2045_Elec_v4_2019",
        "display_name": "Electric",
    },
    "KN2045 H2": {"file_suffix": "KN2045_H2_v4_2019", "display_name": "H2"},
}

# Base path for data files
base_path = (
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/"
)

# Example tech_colors dictionary (replace with your actual colors)
tech_colors = {
    # Example tech colors (replace with your own)
    "BEV charger": "#1f77b4",
    "battery charger": "#ff7f0e",
    "Sonstige": "#2ca02c",
    # Add all your techs here with colors...
}

# Load all scenarios automatically
scenarios = {}
for scenario_name, config in scenario_config.items():
    scenarios[scenario_name] = {
        "daily": pd.read_csv(
            f"{base_path}flex_contribution_d_{config['file_suffix']}.csv", index_col=0
        ),
        "weekly": pd.read_csv(
            f"{base_path}flex_contribution_w_{config['file_suffix']}.csv", index_col=0
        ),
        "display_name": config["display_name"],
    }

# Set up the subplots (1 row, 2 columns)
fig, ax = plt.subplots(1, 2, figsize=(16, 6), sharey=True)


# Function to plot multiple scenarios as grouped stacked bars
def plot_scenarios(ax_subplot, data_dict, title):
    # Get all years from the first scenario
    years = data_dict[list(data_dict.keys())[0]].columns

    # Set up positions for grouped bars
    n_scenarios = len(data_dict)
    n_years = len(years)
    bar_width = 0.8 / n_scenarios
    x_positions = np.arange(n_years)

    # Get all unique technologies across all scenarios for this plot type
    all_technologies = set()
    for scenario_name in data_dict.keys():
        if not data_dict[scenario_name].empty:
            all_technologies.update(data_dict[scenario_name].index)
    all_technologies = sorted(all_technologies)

    # Debug: print technologies found and check colors
    print(f"Technologies found for {title}: {all_technologies}")
    missing_colors = [tech for tech in all_technologies if tech not in tech_colors]
    if missing_colors:
        print(f"Technologies missing colors: {missing_colors}")

    # Plot each scenario
    for i, (scenario_name, df) in enumerate(data_dict.items()):
        # Calculate x positions for this scenario
        x_pos = x_positions + (i - n_scenarios / 2 + 0.5) * bar_width

        # Prepare data for stacking
        bottom = np.zeros(len(years))

        # Plot each technology as part of the stack
        for tech in all_technologies:
            if tech in df.index:
                values = df.loc[tech].values
                # Handle NaN values by replacing with 0
                values = np.nan_to_num(values, nan=0.0)
            else:
                # If technology doesn't exist in this scenario, use zeros
                values = np.zeros(len(years))

            color = tech_colors.get(tech, "gray")
            # Only plot if there are non-zero values
            if np.any(values != 0):
                ax_subplot.bar(
                    x_pos,
                    values,
                    bar_width,
                    bottom=bottom,
                    color=color,
                    label=tech if i == 0 else "",
                )
            bottom += values

    # Customize the subplot
    ax_subplot.set_title(title, fontsize=16)
    ax_subplot.set_xticks(x_positions)
    ax_subplot.set_xticklabels(years, rotation=45)

    # Add scenario legend in upper left with black square markers (does NOT affect bar colors)
    scenario_handles = []
    for scenario_name in data_dict.keys():
        display_name = scenarios[scenario_name]["display_name"]
        handle = plt.Line2D(
            [0],
            [0],
            color="black",
            marker="s",
            linestyle="",
            markersize=8,
            label=display_name,
        )
        scenario_handles.append(handle)
    ax_subplot.legend(handles=scenario_handles, title="Scenarios", loc="upper left")


# Create data dictionaries for daily and weekly
daily_data = {name: info["daily"] for name, info in scenarios.items()}
weekly_data = {name: info["weekly"] for name, info in scenarios.items()}

# Plot daily flexibility needs
plot_scenarios(ax[0], daily_data, "Daily flexibility needs")
ax[0].set_ylabel("TWh / a", fontsize=14)

# Plot weekly flexibility needs
plot_scenarios(ax[1], weekly_data, "Weekly flexibility needs")

# Get all unique technologies for legend
all_technologies_legend = set()
for scenario_info in scenarios.values():
    all_technologies_legend.update(scenario_info["daily"].index)
    all_technologies_legend.update(scenario_info["weekly"].index)
all_technologies_legend = sorted(all_technologies_legend)

# Create the legend handles and labels for all distinct technologies
handles, labels = [], []
for tech in all_technologies_legend:
    handles.append(
        plt.Line2D(
            [0],
            [0],
            marker="s",
            color="w",
            markerfacecolor=tech_colors.get(tech, "gray"),
            markersize=10,
        )
    )
    labels.append(tech if tech != "Sonstige" else "other")

# Place the legend below both subplots
fig.legend(
    handles=handles,
    labels=labels,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.05),
    ncol=5,
    title="flexibility technologies",
)

# Tighten layout to ensure everything fits
plt.tight_layout()

# Define PLOT_DIR (adjust as needed)
PLOT_DIR = "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/plots"

# Save the combined plot
plt.savefig(
    f"{PLOT_DIR}/flexibility_needs_scenarios_combined_english.png", bbox_inches="tight"
)

# Show the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Dictionary to store all dataframes
scenarios = {
    "Mix 1996": {"daily": fc_mix_1996_d, "weekly": fc_mix_1996_w},
    "Mix 2010": {"daily": fc_mix_2010_d, "weekly": fc_mix_2010_w},
    "Mix 2013": {"daily": fc_mix_2013_d, "weekly": fc_mix_2013_w},
    "Mix 2019": {"daily": fc_mix_1996_d, "weekly": fc_mix_1996_w},
}

# Set up the subplots (1 row, 2 columns)
fig, ax = plt.subplots(1, 2, figsize=(16, 6), sharey=True)


# Function to plot multiple scenarios as grouped bars
def plot_scenarios(ax_subplot, data_dict, title):
    # Get all years from the first scenario
    years = data_dict[list(scenarios.keys())[0]].columns

    # Set up positions for grouped bars
    n_scenarios = len(scenarios)
    n_years = len(years)
    bar_width = 0.8 / n_scenarios
    x_positions = np.arange(n_years)

    # Get all unique technologies across all scenarios for this plot type
    all_technologies = set()
    for scenario_name in scenarios.keys():
        if scenario_name in data_dict and not data_dict[scenario_name].empty:
            all_technologies.update(data_dict[scenario_name].index)
    all_technologies = sorted(all_technologies)

    # Debug: print technologies found
    print(f"Technologies found for {title}: {all_technologies}")

    # Plot each scenario
    for i, (scenario_name, df) in enumerate(data_dict.items()):
        # Calculate x positions for this scenario
        x_pos = x_positions + (i - n_scenarios / 2 + 0.5) * bar_width

        # Prepare data for stacking
        bottom = np.zeros(len(years))

        # Plot each technology as part of the stack
        for tech in all_technologies:
            if tech in df.index:
                values = df.loc[tech].values
                # Handle NaN values by replacing with 0
                values = np.nan_to_num(values, nan=0.0)
            else:
                # If technology doesn't exist in this scenario, use zeros
                values = np.zeros(len(years))

            color = tech_colors.get(tech, "gray")

            # Only plot if there are non-zero values
            if np.any(values != 0):
                ax_subplot.bar(
                    x_pos,
                    values,
                    bar_width,
                    bottom=bottom,
                    color=color,
                    label=tech if i == 0 else "",
                )

            bottom += values

    # Customize the subplot
    ax_subplot.set_title(title, fontsize=16)
    ax_subplot.set_xticks(x_positions)
    ax_subplot.set_xticklabels(years, rotation=45)

    # Add scenario labels below x-axis
    scenario_labels = list(scenarios.keys())
    for i, scenario in enumerate(scenario_labels):
        x_pos = x_positions + (i - n_scenarios / 2 + 0.5) * bar_width
        for j, x in enumerate(x_pos):
            if j == len(x_pos) // 2:  # Only label in the middle
                ax_subplot.text(
                    x,
                    ax_subplot.get_ylim()[0]
                    - (ax_subplot.get_ylim()[1] - ax_subplot.get_ylim()[0]) * 0.1,
                    scenario,
                    ha="center",
                    va="top",
                    fontsize=8,
                    rotation=45,
                )


# Create data dictionaries for daily and weekly
daily_data = {name: info["daily"] for name, info in scenarios.items()}
weekly_data = {name: info["weekly"] for name, info in scenarios.items()}

# Plot daily flexibility needs
plot_scenarios(ax[0], daily_data, "Daily flexibility needs")
ax[0].set_ylabel("TWh / a", fontsize=14)

# Plot weekly flexibility needs
plot_scenarios(ax[1], weekly_data, "Weekly flexibility needs")

# Get all unique technologies for legend
all_technologies = set()
for scenario_info in scenarios.values():
    all_technologies.update(scenario_info["daily"].index)
    all_technologies.update(scenario_info["weekly"].index)
all_technologies = sorted(all_technologies)

# Create the legend handles and labels for all distinct technologies
handles, labels = [], []
for tech in all_technologies:
    if tech in tech_colors:
        handles.append(
            plt.Line2D(
                [0],
                [0],
                marker="s",
                color="w",
                markerfacecolor=tech_colors[tech],
                markersize=10,
            )
        )
        labels.append(tech if tech != "Sonstige" else "other")

# Place the legend below both subplots
fig.legend(
    handles=handles,
    labels=labels,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.05),
    ncol=5,
    title="flexibility technologies",
)

# Tighten layout to ensure everything fits
plt.tight_layout()

# Save the combined plot
plt.savefig(
    PLOT_DIR + "/flexibility_needs_weather_years_combined_english", bbox_inches="tight"
)

# Show the plot
plt.show()

#### Cumulative

In [ ]:
df_all_cum_w = pd.DataFrame()

for year in np.arange(2020, 2050, 5):
    df = result_cumulative[year].dropna().diff() * -1
    df_all_cum_w = pd.concat([df_all_cum_w, df], axis=1)

df_all_cum_w

In [ ]:
df = df_all_cum_w.drop(["no"], errors="ignore")

# Set plot properties
fig, ax = plt.subplots(figsize=(8, 5))  # Increase figsize

# Extract colors for the bar plot
colors = [tech_colors[tech] for tech in df.index]

# Create the bar plot
df.T.plot(kind="bar", stacked=True, color=colors, ax=ax)

# Add title and labels
ax.set_title("Wöchentliche Flexibilitätsbedarfe (kumuliert)", fontsize=16)
ax.set_ylabel("TWh / a", fontsize=14)

# Move the legend outside the plot
ax.legend(
    loc="upper left",
    bbox_to_anchor=(1, 1),  # Position the legend outside
    title="Flexibilitätstechnologien",
)

# Tighten the layout to fit everything
plt.tight_layout()

plt.savefig(PLOT_DIR + "/weekly_flex_cumulative")

### Annual

In [ ]:
years = np.arange(2020, 2050, 5)

result_single = pd.DataFrame(index=all_flex_techs, columns=years)
result_cumulative = pd.DataFrame(index=all_flex_techs, columns=years)

for year in years:
    n = networks[year]

    for cumulative in [True, False]:
        if cumulative:
            electricity_supply, electricity_demand = supply_demand(
                n,
                interconnectors=False,
                merge_dist_grid=True,
                drop_dist_grid=False,
                add_diff_as_import=True,
            )
        else:
            electricity_supply, electricity_demand = supply_demand(
                n,
                interconnectors=False,
                merge_dist_grid=True,
                drop_dist_grid=False,
                add_diff_as_import=True,
            )

        non_vre_gens = electricity_supply[
            ~electricity_supply.index.isin(vre_gens)
        ].index

        vre_gen = electricity_supply.loc[vre_gens].sum()
        load = electricity_demand.loc[load_carriers].sum()

        residual_load = load - vre_gen

        for flex_tech in all_flex_techs:
            # 1. Calculate initial residual load
            if flex_tech == "no":
                sub = 0
            elif flex_tech in flex_techs_supply.keys():
                count = sum(
                    item in electricity_supply.index
                    for item in flex_techs_supply[flex_tech]
                )  # how many of the techs are present
                techs = [
                    item
                    for item in flex_techs_supply[flex_tech]
                    if item in electricity_supply.index
                ]  # which techs are present
                if count > 1:
                    sub = electricity_supply.loc[techs].sum()
                elif count == 1:
                    sub = electricity_supply.loc[techs[0]]
                else:
                    print(
                        f"Flex tech {flex_tech} not in electricity supply for year {year}"
                    )
                    continue
            elif flex_tech in flex_techs_demand.keys():
                count = sum(
                    item in electricity_demand.index
                    for item in flex_techs_demand[flex_tech]
                )
                techs = [
                    item
                    for item in flex_techs_demand[flex_tech]
                    if item in electricity_demand.index
                ]
                if count > 1:
                    sub = electricity_demand.loc[techs].sum() * -1
                elif count == 1:
                    sub = electricity_demand.loc[techs[0]] * -1
                else:
                    print(
                        f"Flex tech {flex_tech} not in electricity demand for year {year}"
                    )
                    continue
            else:
                print(f"Flex tech {flex_tech} not in any flex tech list")
                continue

            if cumulative:
                residual_load = residual_load - sub
            else:
                residual_load = load - vre_gen - sub

            # Compute daily average of residual load
            daily_average_residual = residual_load.resample("D").mean()

            # Expand daily averages to hourly resolution for comparison
            daily_average_hourly = daily_average_residual.reindex(
                residual_load.index, method="bfill"
            )

            # Compute weekly average of daily averages
            weekly_average_residual = daily_average_residual.resample("W").mean()
            weekly_average_hourly = weekly_average_residual.reindex(
                residual_load.index, method="bfill"
            )

            # Compute monthly average of weekly averages
            monthly_average_residual = weekly_average_residual.resample("ME").mean()
            monthly_average_hourly = monthly_average_residual.reindex(
                residual_load.index, method="bfill"
            )

            # Compute yearly average of monthly averages
            yearly_average_residual = monthly_average_residual.resample("YE").mean()
            yearly_average_hourly = yearly_average_residual.reindex(
                residual_load.index, method="bfill"
            )

            # Compute the difference between monthly average and yearly (only the positive part)
            yearly_flexibility_needs = (
                monthly_average_hourly - yearly_average_hourly
            ).clip(lower=0)

            # Calculate weekly flexibility needs
            yearly_flexibility_needs = yearly_flexibility_needs.resample("D").sum()

            # Calculate total weekly flexibility needs for one year (TWh/a)
            total_annual_flexibility = yearly_flexibility_needs.sum()

            if cumulative:
                result_cumulative.loc[flex_tech, year] = round(
                    total_annual_flexibility / 1e6, 2
                )
            else:
                result_single.loc[flex_tech, year] = round(
                    total_annual_flexibility / 1e6, 2
                )

#### Single

In [ ]:
df_all_single_a = pd.DataFrame()

for year in np.arange(2020, 2050, 5):
    df = result_single[year]
    no_total_i = df.drop("no").index
    # calc flexibility contribution
    df[no_total_i] = -(result_single[year] - result_single[year].loc["no"])[no_total_i]
    df_all_single_a = pd.concat([df_all_single_a, df], axis=1)

df_all_single_a

In [ ]:
df = df_all_single_a.drop(["no"], errors="ignore")

# Set plot properties
fig, ax = plt.subplots(figsize=(8, 5))  # Increase figsize

# Extract colors for the bar plot
colors = [tech_colors[tech] for tech in df.index]

# Create the bar plot
df.T.plot(kind="bar", stacked=True, color=colors, ax=ax)

# Add title and labels
ax.set_title("Jährliche Flexibilitätsbedarfe", fontsize=16)
ax.set_ylabel("TWh / a", fontsize=14)

# Move the legend outside the plot
ax.legend(
    loc="upper left",
    bbox_to_anchor=(1, 1),  # Position the legend outside
    title="Flexibilitätstechnologien",
)

# Tighten the layout to fit everything
plt.tight_layout()

# Display the plot
plt.show()

In [ ]:
# nice plot
df_single_overall_light_a = df_all_single_a.copy()
df_single_overall_light_a.loc["Import/Export"] = df_single_overall_light_a.loc[
    ["import", "export"]
].sum()
df_single_overall_light_a.loc["PHS"] = df_single_overall_light_a.loc[
    ["PHS charging", "PHS discharging"]
].sum()
df_single_overall_light_a = df_single_overall_light_a.drop(
    [
        "import",
        "export",
        "interconnectors supply",
        "interconnectors demand",
        "PHS charging",
        "PHS discharging",
    ]
)

# drop rows which have less than 1 % of the total flexibility needs in all years
shares = df_single_overall_light_a / df_single_overall_light_a.drop("no").sum()
techs = shares[shares > 0.01].dropna(how="all").index
df_single_overall_light_a = df_single_overall_light_a.loc[techs]
sorted_techs = df_single_overall_light_a.sum(axis=1).sort_values(ascending=False).index
df_single_overall_light_a = df_single_overall_light_a.loc[sorted_techs]
df_single_overall_light_a.loc["Sonstige"] = (
    df_single_overall_light_a.loc["no"] - df_single_overall_light_a.drop("no").sum()
)
df_single_overall_light_a = df_single_overall_light_a.drop("no")

In [ ]:
df = df_single_overall_light_a

# Set plot properties
fig, ax = plt.subplots(figsize=(8, 5))
# Extract colors for the bar plot
colors = [tech_colors[tech] for tech in df.index]

# Create the bar plot
df.T.plot(kind="bar", stacked=True, color=colors, ax=ax)

# Add title and labels
ax.set_title("Jährliche Flexibilitätsbedarfe", fontsize=16)
ax.set_ylabel("TWh / a", fontsize=14)

# Move the legend outside the plot
ax.legend(
    labels=[carriers_in_german.get(l, l) for l in df.index],
    loc="upper left",
    bbox_to_anchor=(1, 1),  # Position the legend outside
    title="Flexibilitätstechnologien",
)

# Tighten the layout to fit everything
plt.tight_layout()

plt.savefig(PLOT_DIR + "/annual_flex_single_nice")

#### Cumulative

In [ ]:
df_all_cum_a = pd.DataFrame()

for year in np.arange(2020, 2050, 5):
    df = result_cumulative[year].dropna().diff() * -1
    df_all_cum_a = pd.concat([df_all_cum_a, df], axis=1)

df_all_cum_a

In [ ]:
df = df_all_cum_a.drop(["no"], errors="ignore")

# Set plot properties
fig, ax = plt.subplots(figsize=(8, 5))  # Increase figsize

# Extract colors for the bar plot
colors = [tech_colors[tech] for tech in df.index]

# Create the bar plot
df.T.plot(kind="bar", stacked=True, color=colors, ax=ax)

# Add title and labels
ax.set_title("Jährliche Flexibilitätsbedarfe", fontsize=16)
ax.set_ylabel("TWh / a", fontsize=14)

# Move the legend outside the plot
ax.legend(
    loc="upper left",
    bbox_to_anchor=(1, 1),  # Position the legend outside
    title="Flexibilitätstechnologien",
    ncol=2,
)

# Tighten the layout to fit everything
plt.tight_layout()

plt.savefig(PLOT_DIR + "/annual_flex_cumulative")

### Flexibility needs overall

In [ ]:
# Stromproduktion in 2045
s, d = supply_demand(
    networks[2045],
    interconnectors=False,
    merge_dist_grid=True,
    drop_dist_grid=False,
    add_diff_as_import=True,
)
s.drop("import", errors="ignore").sum().sum() / 1e6

In [ ]:
240 / (s.drop("import", errors="ignore").sum().sum() / 1e6)

In [ ]:
flex_needs = pd.DataFrame()
flex_needs["daily"] = df_all_single_d.loc["no"]
flex_needs["weekly"] = df_all_single_w.loc["no"]
flex_needs["annual"] = df_all_single_a.loc["no"]
flex_needs

In [ ]:
# Transpose the DataFrame
flex_needs_t = flex_needs.transpose()

# Adjusting for spacing
bar_width = 0.13  # Width of each bar
inner_spacing = 0.02  # Small gap between bars within each category
group_spacing = 0.2  # Larger gap between categories

# Plotting
fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(flex_needs_t))

# Create bars for each year
for i, year in enumerate(flex_needs_t.columns):
    ax.bar(
        [p + i * (bar_width + inner_spacing) + p * group_spacing for p in x],
        flex_needs_t[year],
        width=bar_width,
        label=year,
        color=year_colors[year],
    )

# Configure plot
ax.set_xticks(
    [
        p
        + (len(flex_needs_t.columns) - 1) * (bar_width + inner_spacing) / 2
        + p * group_spacing
        for p in x
    ]
)
ax.set_xticklabels(flex_needs_t.index, rotation=45)
ax.set_ylabel("TWh / a")
ax.legend(title="Jahr")

plt.tight_layout()
plt.savefig(PLOT_DIR + "/flex_needs_overall")

In [ ]:
year_colors_gradient = {
    2020: "#e4c1f9",  # Pale Lavender
    2025: "#c191c9",  # Soft Lilac
    2030: "#9c89b8",  # Lavender Purple
    2035: "#7b68ac",  # Muted Purple
    2040: "#5e4fa2",  # Deep Violet
    2045: "#3c096c",  # Intense Purple
}

# Transpose the DataFrame
flex_needs_t = flex_needs.transpose()

# Adjusting for spacing
bar_width = 0.13  # Width of each bar
inner_spacing = 0.02  # Small gap between bars within each category
group_spacing = 0.2  # Larger gap between categories

# Plotting
fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(flex_needs_t))

# Create bars for each year
for i, year in enumerate(flex_needs_t.columns):
    ax.bar(
        [p + i * (bar_width + inner_spacing) + p * group_spacing for p in x],
        flex_needs_t[year],
        width=bar_width,
        label=year,
        color=year_colors_gradient[year],
    )

# Configure plot
ax.set_xticks(
    [
        p
        + (len(flex_needs_t.columns) - 1) * (bar_width + inner_spacing) / 2
        + p * group_spacing
        for p in x
    ]
)
ax.set_xticklabels(flex_needs_t.index, rotation=45)
ax.set_ylabel("TWh / a")
ax.legend(title="year")

plt.tight_layout()
plt.savefig(PLOT_DIR + "/flex_needs_overall_gradient_english")
plt.show()

In [ ]:
year_colors_gradient = {
    2020: "#e4c1f9",  # Pale Lavender
    2025: "#c191c9",  # Soft Lilac
    2030: "#9c89b8",  # Lavender Purple
    2035: "#7b68ac",  # Muted Purple
    2040: "#5e4fa2",  # Deep Violet
    2045: "#3c096c",  # Intense Purple
}

# Transpose the DataFrame
flex_needs_t = flex_needs.transpose()

# Adjusting for spacing
bar_width = 0.13  # Width of each bar
inner_spacing = 0.02  # Small gap between bars within each category
group_spacing = 0.2  # Larger gap between categories

# Plotting
fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(flex_needs_t))

# Create bars for each year
for i, year in enumerate(flex_needs_t.columns):
    ax.bar(
        [p + i * (bar_width + inner_spacing) + p * group_spacing for p in x],
        flex_needs_t[year],
        width=bar_width,
        label=year,
        color=year_colors_gradient[year],
    )

# Configure plot
ax.set_xticks(
    [
        p
        + (len(flex_needs_t.columns) - 1) * (bar_width + inner_spacing) / 2
        + p * group_spacing
        for p in x
    ]
)
ax.set_xticklabels(flex_needs_t.index, rotation=45)
ax.set_ylabel("TWh / a")
ax.legend(title="year")

plt.tight_layout()
plt.savefig(PLOT_DIR + "/flex_needs_overall_gradient_english")
plt.show()

In [ ]:
fn_1996 = pd.read_csv(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_needs_overall_Mix_1996.csv",
    index_col=0,
)
fn_2010 = pd.read_csv(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_needs_overall_Mix_2010.csv",
    index_col=0,
)
fn_2013 = pd.read_csv(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_needs_overall_Mix_2013.csv",
    index_col=0,
)
fn_2019 = pd.read_csv(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_needs_overall_KN2045_Bal_v4_2019.csv",
    index_col=0,
)

In [ ]:
flex_needs_dict = {1996: fn_1996, 2010: fn_2010, 2013: fn_2013, 2019: fn_2019}

In [ ]:
# Color per weather year (distinct and colorblind-friendly)
weather_colors = {
    1996: "#8dd3c7",  # Soft teal
    2010: "#bebada",  # Soft purple
    2013: "#fb8072",  # Coral
    2019: "#80b1d3",  # Muted blue
}

# Translation of categories
category_labels = {"täglich": "daily", "wöchentlich": "weekly", "jährlich": "yearly"}

# Input
scenario_years = [2020, 2025, 2030, 2035, 2040, 2045]
weather_years = list(flex_needs_dict.keys())
bar_width = 0.2
inner_spacing = 0.02

categories = list(category_labels.keys())  # ['täglich', 'wöchentlich', 'jährlich']

fig, axs = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

x = np.arange(len(scenario_years))  # base x positions for scenario years

for ax, category in zip(axs, categories):
    for i, weather_year in enumerate(weather_years):
        offsets = x + i * (bar_width + inner_spacing)
        heights = [
            flex_needs_dict[weather_year].loc[year, category]
            if year in flex_needs_dict[weather_year].index
            else 0
            for year in scenario_years
        ]

        ax.bar(
            offsets,
            heights,
            width=bar_width,
            label=str(weather_year),
            color=weather_colors[weather_year],
        )

    # X-axis ticks centered
    middle_offset = ((len(weather_years) - 1) * (bar_width + inner_spacing)) / 2
    ax.set_xticks(x + middle_offset)
    ax.set_xticklabels(scenario_years)
    ax.set_title(f"{category_labels[category].capitalize()} flexibility")
    ax.set_xlabel("Scenario Year")

axs[0].set_ylabel("TWh / a")

# Shared legend
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    title="Weather Year",  # Legend title
    loc="center right",  # Position on the right center
    ncol=1,  # Single column
    bbox_to_anchor=(1.12, 0.5),  # Slightly outside right of the plot
)
plt.tight_layout()
plt.subplots_adjust(top=0.85)  # room for legend
plt.savefig(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_needs_combined_weather_years.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
fn_mix = pd.read_csv(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_needs_overall_KN2045_Bal_v4_2019.csv",
    index_col=0,
)
fn_elec = pd.read_csv(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_needs_overall_KN2045_Elec_v4_2019.csv",
    index_col=0,
)
fn_h2 = pd.read_csv(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_needs_overall_KN2045_H2_v4_2019.csv",
    index_col=0,
)
fn_cp = pd.read_csv(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_needs_overall_CurrentPolicies_2019.csv",
    index_col=0,
)

In [ ]:
flex_needs_scenarios = {
    "KN2045_Bal_v4": fn_mix,
    "KN2045_Elec_v4": fn_elec,
    "KN2045_H2_v4": fn_h2,
    "CurrentPolicies": fn_cp,
}

In [ ]:
scenario_colors = {
    "KN2045_Bal_v4": "#66c2a5",  # Teal (balanced)
    "KN2045_Elec_v4": "#fc8d62",  # Orange (electricity-focused)
    "KN2045_H2_v4": "#8da0cb",  # Lavender blue (hydrogen-focused)
    "CurrentPolicies": "#e78ac3",  # Soft pink (status quo)
}

# Translation of categories
category_labels = {"täglich": "daily", "wöchentlich": "weekly", "jährlich": "yearly"}

# Configuration
scenario_years = [2020, 2025, 2030, 2035, 2040, 2045]
scenarios = list(flex_needs_scenarios.keys())
bar_width = 0.2
inner_spacing = 0.02

categories = list(category_labels.keys())

fig, axs = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

x = np.arange(len(scenario_years))  # one per scenario year

for ax, category in zip(axs, categories):
    for i, scenario in enumerate(scenarios):
        offsets = x + i * (bar_width + inner_spacing)
        heights = [
            flex_needs_scenarios[scenario].loc[year, category]
            if year in flex_needs_scenarios[scenario].index
            else 0
            for year in scenario_years
        ]

        ax.bar(
            offsets,
            heights,
            width=bar_width,
            label=scenario,
            color=scenario_colors[scenario],
        )

    # Center xticks
    middle_offset = ((len(scenarios) - 1) * (bar_width + inner_spacing)) / 2
    ax.set_xticks(x + middle_offset)
    ax.set_xticklabels(scenario_years)
    ax.set_title(f"{category_labels[category].capitalize()} flexibility")
    ax.set_xlabel("Scenario Year")

axs[0].set_ylabel("TWh / a")

# Shared legend
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    title="Scenarios",  # Legend title
    loc="center right",  # Position on the right center
    ncol=1,  # Single column
    bbox_to_anchor=(1.12, 0.5),  # Slightly outside right of the plot
)

plt.tight_layout()
plt.subplots_adjust(top=0.85)
plt.savefig(
    "/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_needs_by_scenario_subplots.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
year_colors_gradient = {
    2020: "#e4c1f9",  # Pale Lavender
    2025: "#c191c9",  # Soft Lilac
    2030: "#9c89b8",  # Lavender Purple
    2035: "#7b68ac",  # Muted Purple
    2040: "#5e4fa2",  # Deep Violet
    2045: "#3c096c",  # Intense Purple
}

# Transpose the DataFrame
flex_needs.columns = ["täglich", "wöchentlich", "jährlich"]
flex_needs_t = flex_needs.transpose()


# Adjusting for spacing
bar_width = 0.13  # Width of each bar
inner_spacing = 0.02  # Small gap between bars within each category
group_spacing = 0.2  # Larger gap between categories

# Plotting
fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(flex_needs_t))

# Create bars for each year
for i, year in enumerate(flex_needs_t.columns):
    ax.bar(
        [p + i * (bar_width + inner_spacing) + p * group_spacing for p in x],
        flex_needs_t[year],
        width=bar_width,
        label=year,
        color=year_colors_gradient[year],
    )

# Configure plot
ax.set_xticks(
    [
        p
        + (len(flex_needs_t.columns) - 1) * (bar_width + inner_spacing) / 2
        + p * group_spacing
        for p in x
    ]
)
ax.set_xticklabels(flex_needs_t.index, rotation=45, fontsize=14)
ax.set_title("Flexibilitätsbedarfe verschiedener Zeitskalen [TWh/a]", fontsize=16)
ax.set_ylabel("TWh/a", fontsize=14)
ax.legend(title="Jahr", fontsize=12)

plt.tight_layout()
plt.savefig(PLOT_DIR + "/flex_needs_overall_gradient.png")
plt.savefig(PLOT_DIR + "/flex_needs_overall_gradient.pdf")
plt.show()

# save df
flex_needs.to_csv(
    f"/home/julian-geis/repos/01_pricing-paper/pricing_analysis/data/results/flex_plots/flex_needs_overall_{scenario}_{weather_year}.csv"
)

In [ ]:
assert 0

In [ ]:
from ecmwfapi import ECMWFDataServer

server = ECMWFDataServer()

server.retrieve(
    {
        "class": "ti",
        "dataset": "tigge",
        "date": "2019-01-01/to/2019-01-31",
        "expver": "prod",
        "grid": "0.5/0.5",
        "levtype": "sfc",
        "number": "1",
        "origin": "kwbc",
        "param": "166/177",
        "step": "0",
        "time": "00:00:00/06:00:00/12:00:00/18:00:00",
        "type": "pf",
        "target": "output",
    }
)

## Flexibility by supply and demand (not aggregated)

old problems:
- how would you see the flexibility contribution of a technology that does only shift between the buses (AC+DC) if you aggregate all buses?

new problems:
- how do you see the flexibility contribution of a technology to the overall DE residual load if you only see the residual load of the bus (OCGT plant providing electricity for another bus)


In [ ]:
# Define flexible technologies
flex_techs_supply = {
    "no": "",
    "interconnectors supply": ["AC", "DC"],
    "coal": ["lignite", "coal"],
    "gas": ["OCGT", "CCGT"],
    "biogas": ["biogas"],
    "oil": ["oil"],
    "nuclear": ["nuclear"],
    "solid biomass": ["solid biomass"],
    "PHS discharging": ["PHS"],
    "H2 OCGT": ["H2 OCGT", "H2 retrofit OCGT"],
    "battery discharger": ["battery discharger", "home battery discharger"],
    "coal CHP": ["urban central coal CHP", "urban central lignite CHP"],
    "gas CHP": ["urban central gas CHP", "urban central gas CHP CC"],
    "H2 CHP": ["urban central H2 CHP", "urban central H2 retrofit CHP"],
    "biomass CHP": [
        "urban central solid biomass CHP",
        "urban central solid biomass CHP CC",
    ],
    "waste CHP": ["waste CHP", "waste CHP CC"],
    "hydro": ["hydro"],
    "Fuel Cell": ["H2 Fuel Cell"],
}

flex_techs_demand = {
    "interconnectors demand": ["AC", "DC"],
    "H2 Electrolysis": ["H2 Electrolysis"],
    "BEV charger": ["BEV charger"],
    "PHS charging": ["PHS"],
    "battery charger": ["battery charger", "home battery charger"],
    "heat pump": [
        "rural ground heat pump",
        "rural air heat pump",
        "urban decentral air heat pump",
        "urban central air heat pump",
    ],
    "resistive heater": [
        "rural resistive heater",
        "urban decentral resistive heater",
        "urban central resistive heater",
    ],
    "methanolisation": ["methanolisation"],
}

all_flex_techs = list(flex_techs_supply.keys()) + list(flex_techs_demand.keys())
bus_regions_de = n.buses.location[n.buses.location.str.contains("DE")].unique().tolist()

In [ ]:
n = networks[2045]
s, d = supply_demand(
    n, interconnectors=True, merge_dist_grid=True, drop_dist_grid=False
)

In [ ]:
locs = n.buses.location[n.buses.location.str.contains("DE")].unique().tolist()
loc = locs[0]

bus_strings = []
for loc in locs:
    bus_A = n.buses[(n.buses.carrier == "AC") & (n.buses.location == loc)].index
    bus_B = n.buses[
        (n.buses.carrier == "low voltage") & (n.buses.location == loc)
    ].index

    bus_strings.append(list(map(str, list(bus_A) + list(bus_B))))

### Daily

In [ ]:
# simple testing
n = networks[2045]
electricity_supply, electricity_demand = supply_demand(
    n,
    interconnectors=True,
    merge_dist_grid=True,
    drop_dist_grid=False,
    buses=bus_strings[1],
)

non_vre_gens = electricity_supply[~electricity_supply.index.isin(vre_gens)].index
vre_gens_s = list(set(vre_gens) & set(electricity_supply.index))

vre_gen = electricity_supply.loc[vre_gens_s].sum()
load = electricity_demand.loc[load_carriers].sum()
demand = electricity_demand.sum()

residual_load = (
    load - vre_gen
)  # - electricity_supply.loc["oil"]# - electricity_supply.loc["DC"]#+ electricity_demand.loc["AC"] #- electricity_supply.loc["PHS"]

# Compute daily average of residual load
daily_average_residual = residual_load.resample("D").mean()

# Expand daily averages to hourly resolution for comparison
daily_average_hourly = daily_average_residual.reindex(
    residual_load.index, method="ffill"
)

# Compute the difference between residual load and daily average (only the positive part)
hourly_flexibility_needs = (residual_load - daily_average_hourly).clip(lower=0)

# Calculate daily flexibility needs
daily_flexibility_needs = hourly_flexibility_needs.resample("D").sum()

# Calculate total annual flexibility needs
total_annual_flexibility = hourly_flexibility_needs.sum()

print(f"Total annual flexibility needs: {round(total_annual_flexibility / 1e6, 2)} TWh")

In [ ]:
# 25 min for all buses single + cumulative

years = np.arange(2020, 2050, 5)

result_single_all_buses = {}
result_cumulative_all_buses = {}


load_carriers = ["electricity", "agriculture electricity", "industry electricity"]
vre_gens = [
    "onwind",
    "offwind-ac",
    "offwind-dc",
    "solar",
    "solar-hsat",
    "solar rooftop",
    "ror",
]

for buses in bus_strings:
    result_single = pd.DataFrame(index=all_flex_techs, columns=years)
    result_cumulative = pd.DataFrame(index=all_flex_techs, columns=years)

    for year in years:  # [2020,2040,2045]:
        n = networks[year]

        for cumulative in [True, False]:
            if cumulative:
                electricity_supply, electricity_demand = supply_demand(
                    n,
                    interconnectors=True,
                    merge_dist_grid=True,
                    drop_dist_grid=False,
                    buses=buses,
                )
            else:
                electricity_supply, electricity_demand = supply_demand(
                    n,
                    interconnectors=False,
                    merge_dist_grid=True,
                    drop_dist_grid=False,
                    buses=buses,
                )

            vre_gens_s = list(set(vre_gens) & set(electricity_supply.index))
            non_vre_gens = electricity_supply[
                ~electricity_supply.index.isin(vre_gens)
            ].index

            vre_gen = electricity_supply.loc[vre_gens_s].sum()
            load = electricity_demand.loc[load_carriers].sum()

            residual_load = load - vre_gen

            for flex_tech in all_flex_techs:
                # 1. Calculate initial residual load
                if flex_tech == "no":
                    sub = 0
                elif flex_tech in flex_techs_supply.keys():
                    count = sum(
                        item in electricity_supply.index
                        for item in flex_techs_supply[flex_tech]
                    )  # how many of the techs are present
                    techs = [
                        item
                        for item in flex_techs_supply[flex_tech]
                        if item in electricity_supply.index
                    ]  # which techs are present
                    if count > 1:
                        sub = electricity_supply.loc[techs].sum()
                    elif count == 1:
                        sub = electricity_supply.loc[techs[0]]
                    else:
                        print(
                            f"Flex tech {flex_tech} not in electricity supply for year {year}"
                        )
                        continue
                elif flex_tech in flex_techs_demand.keys():
                    count = sum(
                        item in electricity_demand.index
                        for item in flex_techs_demand[flex_tech]
                    )
                    techs = [
                        item
                        for item in flex_techs_demand[flex_tech]
                        if item in electricity_demand.index
                    ]
                    if count > 1:
                        sub = electricity_demand.loc[techs].sum() * -1
                    elif count == 1:
                        sub = electricity_demand.loc[techs[0]] * -1
                    else:
                        print(
                            f"Flex tech {flex_tech} not in electricity demand for year {year}"
                        )
                        continue
                else:
                    print(f"Flex tech {flex_tech} not in any flex tech list")
                    continue

                if cumulative:
                    residual_load = residual_load - sub
                else:
                    residual_load = demand - vre_gen - sub

                # 2. Compute daily average of residual load
                daily_average_residual = residual_load.resample("D").mean()

                # Expand daily averages to hourly resolution for comparison
                daily_average_hourly = daily_average_residual.reindex(
                    residual_load.index, method="ffill"
                )

                # 3. Compute the difference between residual load and daily average (only the positive part)
                hourly_flexibility_needs = (residual_load - daily_average_hourly).clip(
                    lower=0
                )

                # Calculate daily flexibility needs
                daily_flexibility_needs = hourly_flexibility_needs.resample("D").sum()

                # 4. Calculate total annual flexibility needs
                total_annual_flexibility = hourly_flexibility_needs.sum()

                if cumulative:
                    result_cumulative.loc[flex_tech, year] = round(
                        total_annual_flexibility / 1e6, 2
                    )
                else:
                    result_single.loc[flex_tech, year] = round(
                        total_annual_flexibility / 1e6, 2
                    )

        result_single_all_buses[buses[0]] = result_single
        result_cumulative_all_buses[buses[0]] = result_cumulative

- if only one buses argument is selected but all years it seems correct
- if more than one buses arg is selected all entries are wrong

#### cumulative 

In [ ]:
# calc technology contribution for every region
result_cumulative_all_buses_diff = {}

for bus_region in bus_regions_de:
    df_all_cum = pd.DataFrame()
    for year in np.arange(2020, 2050, 5):
        df = result_cumulative_all_buses[bus_region][year].dropna().diff() * -1
        df_all_cum = pd.concat([df_all_cum, df], axis=1)

    result_cumulative_all_buses_diff[bus_region] = df_all_cum

In [ ]:
# aggregate all regions
df_cum_overall = pd.DataFrame(index=all_flex_techs, columns=years, dtype=float).fillna(
    0.0
)

for bus_region in bus_regions_de:
    df = result_cumulative_all_buses_diff[bus_region]
    df = df.astype(float)
    df_cum_overall.loc[df.index] = df_cum_overall.loc[df.index].add(df, fill_value=0.0)

df_cum_overall_daily_b = df_cum_overall

In [ ]:
df = (df_cum_overall_daily_b * 1e3 / 365).drop(["no"], errors="ignore")


# Set plot properties
fig, ax = plt.subplots(figsize=(8, 5))  # Increase figsize

# Extract colors for the bar plot
colors = [tech_colors[tech] for tech in df.index]

# Create the bar plot
df.T.plot(kind="bar", stacked=True, color=colors, ax=ax)

# Add title and labels
ax.set_title("Tägliche Flexibilitätsbedarfe", fontsize=16)
ax.set_ylabel("GWh / Tag", fontsize=14)

# Move the legend outside the plot
ax.legend(
    loc="upper left",
    bbox_to_anchor=(1, 1),  # Position the legend outside
    title="Flexibilitätstechnologien",
)

# Tighten the layout to fit everything
plt.tight_layout()

# Display the plot
plt.show()

In [ ]:
df_cum_overall_light = df_cum_overall_daily_b.copy()
df_cum_overall_light.loc["Stromnetz"] = df_cum_overall_light.loc[
    ["interconnectors supply", "interconnectors demand"]
].sum()
df_cum_overall_light.loc["PHS"] = df_cum_overall_light.loc[
    ["PHS charging", "PHS discharging"]
].sum()
df_cum_overall_light = df_cum_overall_light.drop(
    [
        "no",
        "interconnectors supply",
        "interconnectors demand",
        "PHS charging",
        "PHS discharging",
    ]
)
# drop rows which have less than 1 % of the total flexibility needs in all years

shares = df_cum_overall_light / df_cum_overall_light.sum()
techs = shares[shares > 0.01].dropna(how="all").index
df_cum_overall_light = df_cum_overall_light.loc[techs]
sorted_techs = df_cum_overall_light.sum(axis=1).sort_values(ascending=False).index
df_cum_overall_light = df_cum_overall_light.loc[sorted_techs]

In [ ]:
df_cum_overall_light

In [ ]:
df = df_cum_overall_light * 1e3 / 365
tech_colors["Stromnetz"] = "dimgrey"
tech_colors["battery discharger"] = "palegreen"
tech_colors["resistive heater"] = "khaki"

# Set plot properties
fig, ax = plt.subplots(figsize=(8, 5))
# Extract colors for the bar plot
colors = [tech_colors[tech] for tech in df.index]

# Create the bar plot
df.T.plot(kind="bar", stacked=True, color=colors, ax=ax)

# Add title and labels
ax.set_title("Tägliche Flexibilitätsbedarfe", fontsize=16)
ax.set_ylabel("GWh / Tag", fontsize=14)

# Move the legend outside the plot
ax.legend(
    labels=[carriers_in_german.get(l, l) for l in df.index],
    loc="upper left",
    bbox_to_anchor=(1, 1),  # Position the legend outside
    title="Flexibilitätstechnologien",
)

# Tighten the layout to fit everything
plt.tight_layout()

# Display the plot
plt.show()

#### single

In [ ]:
# calc technology contribution for every region
result_single_all_buses_diff = {}

for bus_region in bus_regions_de:
    df_all_single = pd.DataFrame()
    for year in np.arange(2020, 2050, 5):
        df = result_single_all_buses[bus_region][year].copy()
        no_total_i = df.drop("no").index
        df[no_total_i] = -(df - df.loc["no"])[no_total_i]
        df_all_single = pd.concat([df_all_single, df], axis=1)

    result_single_all_buses_diff[bus_region] = df_all_single

In [ ]:
# aggregate all regions
df_single_overall = pd.DataFrame(
    index=all_flex_techs, columns=years, dtype=float
).fillna(0.0)

for bus_region in bus_regions_de:
    df = result_single_all_buses_diff[bus_region]
    df = df.astype(float)
    df_single_overall.loc[df.index] = df_single_overall.loc[df.index].add(
        df, fill_value=0.0
    )

df_single_overall_daily_b = df_single_overall

In [ ]:
df_single_overall

In [ ]:
df = df_single_overall_daily_b.drop(["no"], errors="ignore")

# Set plot properties
fig, ax = plt.subplots(figsize=(8, 5))  # Increase figsize

# Extract colors for the bar plot
colors = [tech_colors[tech] for tech in df.index]

# Create the bar plot
df.T.plot(kind="bar", stacked=True, color=colors, ax=ax)

# Add title and labels
ax.set_title("Tägliche Flexibilitätsbedarfe", fontsize=16)
ax.set_ylabel("GWh / Tag", fontsize=14)

# Move the legend outside the plot
ax.legend(
    loc="upper left",
    bbox_to_anchor=(1, 1),  # Position the legend outside
    title="Flexibilitätstechnologien",
)

# Tighten the layout to fit everything
plt.tight_layout()

# Display the plot
plt.show()

### Weekly

In [ ]:
# simple testing
n = networks[2045]

# local elec supply / demand at bus level
electricity_supply, electricity_demand = supply_demand(
    n,
    interconnectors=True,
    merge_dist_grid=True,
    drop_dist_grid=False,
    buses=["DE0 20", "DE0 20 low voltage"],
)  # bus_strings[5])

non_vre_gens = electricity_supply[~electricity_supply.index.isin(vre_gens)].index
vre_gens_s = list(set(vre_gens) & set(electricity_supply.index))

vre_gen = electricity_supply.loc[vre_gens_s].sum()
load = electricity_demand.loc[load_carriers].sum()

residual_load = (
    load - vre_gen - electricity_supply.loc["AC"]
)  # electricity_supply.loc["urban central gas CHP"]# - electricity_supply.loc["DC"]#+ electricity_demand.loc["AC"] #- electricity_supply.loc["PHS"]

# overall elec supply / demand in Germany
electricity_supply_de, electricity_demand_de = supply_demand(
    n, interconnectors=True, merge_dist_grid=True, drop_dist_grid=False
)

non_vre_gens = electricity_supply_de[~electricity_supply_de.index.isin(vre_gens)].index
vre_gens_s = list(set(vre_gens) & set(electricity_supply.index))

vre_gen_de = electricity_supply_de.loc[vre_gens_s].sum()
load_de = electricity_demand_de.loc[load_carriers].sum()

residual_load_de = (
    load_de
    - vre_gen_de
    - electricity_supply_de.loc["AC"]
    + electricity_demand_de.loc["AC"]
    - electricity_supply.loc["H2 OCGT"]
)  # + electricity_demand.loc["AC"] #- electricity_supply.loc["PHS"]
residual_load = residual_load_de

# Compute daily average of residual load
daily_average_residual = residual_load.resample("D").mean()

# Expand daily averages to hourly resolution for comparison
daily_average_hourly = daily_average_residual.reindex(
    residual_load.index, method="ffill"
)

# Compute weekly average of daily averages
weekly_average_residual = daily_average_residual.resample("W").mean()
weekly_average_hourly = weekly_average_residual.reindex(
    residual_load.index, method="ffill"
)

# Compute the difference between daily average and weekly average (only the positive part)
weekly_flexibility_needs = (daily_average_hourly - weekly_average_hourly).clip(lower=0)

# Calculate weekly flexibility needs
weekly_flexibility_needs = weekly_flexibility_needs.resample("W").sum()

# Calculate total weekly flexibility needs for one year (TWh/a)
total_annual_flexibility = weekly_flexibility_needs.sum()

print(f"Total annual flexibility needs: {round(total_annual_flexibility / 1e6, 2)} TWh")
# 108.15 TWh

In [ ]:
# 20 min fo all buses and single + cumulative

years = np.arange(2020, 2050, 5)

result_single_all_buses = {}
result_cumulative_all_buses = {}


load_carriers = ["electricity", "agriculture electricity", "industry electricity"]
vre_gens = [
    "onwind",
    "offwind-ac",
    "offwind-dc",
    "solar",
    "solar-hsat",
    "solar rooftop",
    "ror",
]

for buses in bus_strings:
    result_single = pd.DataFrame(index=all_flex_techs, columns=years)
    result_cumulative = pd.DataFrame(index=all_flex_techs, columns=years)

    for year in years:  # [2020,2040,2045]:
        n = networks[year]

        for cumulative in [True, False]:
            if cumulative:
                electricity_supply, electricity_demand = supply_demand(
                    n,
                    interconnectors=True,
                    merge_dist_grid=True,
                    drop_dist_grid=False,
                    buses=buses,
                )
            else:
                electricity_supply, electricity_demand = supply_demand(
                    n,
                    interconnectors=False,
                    merge_dist_grid=True,
                    drop_dist_grid=False,
                    buses=buses,
                )

            vre_gens_s = list(set(vre_gens) & set(electricity_supply.index))
            non_vre_gens = electricity_supply[
                ~electricity_supply.index.isin(vre_gens)
            ].index

            vre_gen = electricity_supply.loc[vre_gens_s].sum()
            load = electricity_demand.loc[load_carriers].sum()

            residual_load = load - vre_gen

            for flex_tech in all_flex_techs:
                # 1. Calculate initial residual load
                if flex_tech == "no":
                    sub = 0
                elif flex_tech in flex_techs_supply.keys():
                    count = sum(
                        item in electricity_supply.index
                        for item in flex_techs_supply[flex_tech]
                    )  # how many of the techs are present
                    techs = [
                        item
                        for item in flex_techs_supply[flex_tech]
                        if item in electricity_supply.index
                    ]  # which techs are present
                    if count > 1:
                        sub = electricity_supply.loc[techs].sum()
                    elif count == 1:
                        sub = electricity_supply.loc[techs[0]]
                    else:
                        print(
                            f"Flex tech {flex_tech} not in electricity supply for year {year}"
                        )
                        continue
                elif flex_tech in flex_techs_demand.keys():
                    count = sum(
                        item in electricity_demand.index
                        for item in flex_techs_demand[flex_tech]
                    )
                    techs = [
                        item
                        for item in flex_techs_demand[flex_tech]
                        if item in electricity_demand.index
                    ]
                    if count > 1:
                        sub = electricity_demand.loc[techs].sum() * -1
                    elif count == 1:
                        sub = electricity_demand.loc[techs[0]] * -1
                    else:
                        print(
                            f"Flex tech {flex_tech} not in electricity demand for year {year}"
                        )
                        continue
                else:
                    print(f"Flex tech {flex_tech} not in any flex tech list")
                    continue

                if cumulative:
                    residual_load = residual_load - sub
                else:
                    residual_load = demand - vre_gen - sub

                # Compute daily average of residual load
                daily_average_residual = residual_load.resample("D").mean()

                # Expand daily averages to hourly resolution for comparison
                daily_average_hourly = daily_average_residual.reindex(
                    residual_load.index, method="ffill"
                )

                # Compute weekly average of daily averages
                weekly_average_residual = daily_average_residual.resample("W").mean()
                weekly_average_hourly = weekly_average_residual.reindex(
                    residual_load.index, method="ffill"
                )

                # Compute the difference between daily average and weekly average (only the positive part)
                weekly_flexibility_needs = (
                    daily_average_hourly - weekly_average_hourly
                ).clip(lower=0)

                # Calculate weekly flexibility needs
                weekly_flexibility_needs = weekly_flexibility_needs.resample("W").sum()

                # Calculate total weekly flexibility needs for one year (TWh/a)
                total_annual_flexibility = weekly_flexibility_needs.sum()

                if cumulative:
                    result_cumulative.loc[flex_tech, year] = round(
                        total_annual_flexibility / 1e6, 2
                    )
                else:
                    result_single.loc[flex_tech, year] = round(
                        total_annual_flexibility / 1e6, 2
                    )

        result_single_all_buses[buses[0]] = result_single
        result_cumulative_all_buses[buses[0]] = result_cumulative

#### cumulative

In [ ]:
# calc technology contribution for every region
result_cumulative_all_buses_diff = {}

for bus_region in bus_regions_de:
    df_all_cum = pd.DataFrame()
    for year in np.arange(2020, 2050, 5):
        df = result_cumulative_all_buses[bus_region][year].dropna().diff() * -1
        df_all_cum = pd.concat([df_all_cum, df], axis=1)

    result_cumulative_all_buses_diff[bus_region] = df_all_cum

# aggregate all regions
df_cum_overall = pd.DataFrame(index=all_flex_techs, columns=years, dtype=float).fillna(
    0.0
)

for bus_region in bus_regions_de:
    df = result_cumulative_all_buses_diff[bus_region]
    df = df.astype(float)
    df_cum_overall.loc[df.index] = df_cum_overall.loc[df.index].add(df, fill_value=0.0)

df_cum_overall_weekly_b = df_cum_overall

In [ ]:
df_cum_overall_weekly_b

In [ ]:
df = (df_cum_overall_weekly_b / 52).drop(["no"], errors="ignore")

# Set plot properties
fig, ax = plt.subplots(figsize=(8, 5))  # Increase figsize

# Extract colors for the bar plot
colors = [tech_colors[tech] for tech in df.index]

# Create the bar plot
df.T.plot(kind="bar", stacked=True, color=colors, ax=ax)

# Add title and labels
ax.set_title("Wöchentliche Flexibilitätsbedarfe", fontsize=16)
ax.set_ylabel("TWh / Woche", fontsize=14)

# Move the legend outside the plot
ax.legend(
    loc="upper left",
    bbox_to_anchor=(1, 1),  # Position the legend outside
    title="Flexibilitätstechnologien",
)

# Tighten the layout to fit everything
plt.tight_layout()

# Display the plot
plt.show()

In [ ]:
df_cum_overall_weekly_b_light = df_cum_overall_weekly_b.copy()
df_cum_overall_weekly_b_light.loc["Stromnetz"] = df_cum_overall_weekly_b_light.loc[
    ["interconnectors supply", "interconnectors demand"]
].sum()
df_cum_overall_weekly_b_light.loc["PHS"] = df_cum_overall_weekly_b_light.loc[
    ["PHS charging", "PHS discharging"]
].sum()
v_light = df_cum_overall_weekly_b_light.drop(
    [
        "no",
        "interconnectors supply",
        "interconnectors demand",
        "PHS charging",
        "PHS discharging",
    ]
)
# drop rows which have less than 1 % of the total flexibility needs in all years

shares = v_light / df_cum_overall_weekly_b_light.sum()
techs = shares[shares > 0.005].dropna(how="all").index
df_cum_overall_weekly_b_light = df_cum_overall_weekly_b_light.loc[techs]
sorted_techs = (
    df_cum_overall_weekly_b_light.sum(axis=1).sort_values(ascending=False).index
)
df_cum_overall_weekly_b_light = df_cum_overall_weekly_b_light.loc[sorted_techs]

In [ ]:
df_cum_overall_weekly_b_light

In [ ]:
df = (df_cum_overall_weekly_b_light / 52).drop(["no"], errors="ignore")

tech_colors["Stromnetz"] = "dimgrey"
tech_colors["battery discharger"] = "palegreen"
tech_colors["resistive heater"] = "khaki"

# Set plot properties
fig, ax = plt.subplots(figsize=(8, 5))  # Increase figsize

# Extract colors for the bar plot
colors = [tech_colors[tech] for tech in df.index]

# Create the bar plot
df.T.plot(kind="bar", stacked=True, color=colors, ax=ax)

# Add title and labels
ax.set_title("Wöchentliche Flexibilitätsbedarfe", fontsize=16)
ax.set_ylabel("TWh / Woche", fontsize=14)

# Move the legend outside the plot
ax.legend(
    labels=[carriers_in_german.get(l, l) for l in df.index],
    loc="upper left",
    bbox_to_anchor=(1, 1),  # Position the legend outside
    title="Flexibilitätstechnologien",
)

# Tighten the layout to fit everything
plt.tight_layout()

# Display the plot
plt.show()

#### single

In [ ]:
# calc technology contribution for every region
result_single_all_buses_diff = {}

for bus_region in bus_regions_de:
    df_all_single = pd.DataFrame()
    for year in np.arange(2020, 2050, 5):
        df = result_single_all_buses[bus_region][year].copy()
        no_total_i = df.drop("no").index
        df[no_total_i] = -(df - df.loc["no"])[no_total_i]
        df_all_single = pd.concat([df_all_single, df], axis=1)

    result_single_all_buses_diff[bus_region] = df_all_single

# aggregate all regions
df_single_overall = pd.DataFrame(
    index=all_flex_techs, columns=years, dtype=float
).fillna(0.0)

for bus_region in bus_regions_de:
    df = result_single_all_buses_diff[bus_region]
    df = df.astype(float)
    df_single_overall.loc[df.index] = df_single_overall.loc[df.index].add(
        df, fill_value=0.0
    )

df_single_overall_weekly_b = df_single_overall

In [ ]:
df = df_single_overall_weekly_b.drop(["no"], errors="ignore")

# Set plot properties
fig, ax = plt.subplots(figsize=(8, 5))  # Increase figsize

# Extract colors for the bar plot
colors = [tech_colors[tech] for tech in df.index]

# Create the bar plot
df.T.plot(kind="bar", stacked=True, color=colors, ax=ax)

# Add title and labels
ax.set_title("Wöchentliche Flexibilitätsbedarfe", fontsize=16)
ax.set_ylabel("TWh", fontsize=14)

# Move the legend outside the plot
ax.legend(
    loc="upper left",
    bbox_to_anchor=(1, 1),  # Position the legend outside
    title="Flexibilitätstechnologien",
)

# Tighten the layout to fit everything
plt.tight_layout()

# Display the plot
plt.show()

### Annual

In [ ]:
# 20 min fo all buses and single + cumulative

years = np.arange(2020, 2050, 5)

result_single_all_buses = {}
result_cumulative_all_buses = {}


load_carriers = ["electricity", "agriculture electricity", "industry electricity"]
vre_gens = [
    "onwind",
    "offwind-ac",
    "offwind-dc",
    "solar",
    "solar-hsat",
    "solar rooftop",
    "ror",
]

for buses in bus_strings:
    result_single = pd.DataFrame(index=all_flex_techs, columns=years)
    result_cumulative = pd.DataFrame(index=all_flex_techs, columns=years)

    for year in years:  # [2020,2040,2045]:
        n = networks[year]

        for cumulative in [True, False]:
            if cumulative:
                electricity_supply, electricity_demand = supply_demand(
                    n,
                    interconnectors=True,
                    merge_dist_grid=True,
                    drop_dist_grid=False,
                    buses=buses,
                )
            else:
                electricity_supply, electricity_demand = supply_demand(
                    n,
                    interconnectors=False,
                    merge_dist_grid=True,
                    drop_dist_grid=False,
                    buses=buses,
                )

            vre_gens_s = list(set(vre_gens) & set(electricity_supply.index))
            non_vre_gens = electricity_supply[
                ~electricity_supply.index.isin(vre_gens)
            ].index

            vre_gen = electricity_supply.loc[vre_gens_s].sum()
            load = electricity_demand.loc[load_carriers].sum()

            residual_load = load - vre_gen

            for flex_tech in all_flex_techs:
                # 1. Calculate initial residual load
                if flex_tech == "no":
                    sub = 0
                elif flex_tech in flex_techs_supply.keys():
                    count = sum(
                        item in electricity_supply.index
                        for item in flex_techs_supply[flex_tech]
                    )  # how many of the techs are present
                    techs = [
                        item
                        for item in flex_techs_supply[flex_tech]
                        if item in electricity_supply.index
                    ]  # which techs are present
                    if count > 1:
                        sub = electricity_supply.loc[techs].sum()
                    elif count == 1:
                        sub = electricity_supply.loc[techs[0]]
                    else:
                        print(
                            f"Flex tech {flex_tech} not in electricity supply for year {year}"
                        )
                        continue
                elif flex_tech in flex_techs_demand.keys():
                    count = sum(
                        item in electricity_demand.index
                        for item in flex_techs_demand[flex_tech]
                    )
                    techs = [
                        item
                        for item in flex_techs_demand[flex_tech]
                        if item in electricity_demand.index
                    ]
                    if count > 1:
                        sub = electricity_demand.loc[techs].sum() * -1
                    elif count == 1:
                        sub = electricity_demand.loc[techs[0]] * -1
                    else:
                        print(
                            f"Flex tech {flex_tech} not in electricity demand for year {year}"
                        )
                        continue
                else:
                    print(f"Flex tech {flex_tech} not in any flex tech list")
                    continue

                if cumulative:
                    residual_load = residual_load - sub
                else:
                    residual_load = demand - vre_gen - sub

                # Compute daily average of residual load
                daily_average_residual = residual_load.resample("D").mean()

                # Expand daily averages to hourly resolution for comparison
                daily_average_hourly = daily_average_residual.reindex(
                    residual_load.index, method="bfill"
                )

                # Compute weekly average of daily averages
                weekly_average_residual = daily_average_residual.resample("W").mean()
                weekly_average_hourly = weekly_average_residual.reindex(
                    residual_load.index, method="bfill"
                )

                # Compute monthly average of weekly averages
                monthly_average_residual = weekly_average_residual.resample("ME").mean()
                monthly_average_hourly = monthly_average_residual.reindex(
                    residual_load.index, method="bfill"
                )

                # Compute yearly average of monthly averages
                yearly_average_residual = monthly_average_residual.resample("YE").mean()
                yearly_average_hourly = yearly_average_residual.reindex(
                    residual_load.index, method="bfill"
                )

                # Compute the difference between monthly average and yearly (only the positive part)
                yearly_flexibility_needs = (
                    monthly_average_hourly - yearly_average_hourly
                ).clip(lower=0)

                # Calculate weekly flexibility needs
                yearly_flexibility_needs = yearly_flexibility_needs.resample("D").sum()

                # Calculate total weekly flexibility needs for one year (TWh/a)
                total_annual_flexibility = yearly_flexibility_needs.sum()

                if cumulative:
                    result_cumulative.loc[flex_tech, year] = round(
                        total_annual_flexibility / 1e6, 2
                    )
                else:
                    result_single.loc[flex_tech, year] = round(
                        total_annual_flexibility / 1e6, 2
                    )

        result_single_all_buses[buses[0]] = result_single
        result_cumulative_all_buses[buses[0]] = result_cumulative

#### cumulative

In [ ]:
# calc technology contribution for every region
result_cumulative_all_buses_diff = {}

for bus_region in bus_regions_de:
    df_all_cum = pd.DataFrame()
    for year in np.arange(2020, 2050, 5):
        df = result_cumulative_all_buses[bus_region][year].dropna().diff() * -1
        df_all_cum = pd.concat([df_all_cum, df], axis=1)

    result_cumulative_all_buses_diff[bus_region] = df_all_cum

# aggregate all regions
df_cum_overall = pd.DataFrame(index=all_flex_techs, columns=years, dtype=float).fillna(
    0.0
)

for bus_region in bus_regions_de:
    df = result_cumulative_all_buses_diff[bus_region]
    df = df.astype(float)
    df_cum_overall.loc[df.index] = df_cum_overall.loc[df.index].add(df, fill_value=0.0)

df_cum_overall_annual_b = df_cum_overall

In [ ]:
df = df_cum_overall_annual_b.drop(["no"], errors="ignore")

# Set plot properties
fig, ax = plt.subplots(figsize=(8, 5))  # Increase figsize

# Extract colors for the bar plot
colors = [tech_colors[tech] for tech in df.index]

# Create the bar plot
df.T.plot(kind="bar", stacked=True, color=colors, ax=ax)

# Add title and labels
ax.set_title("Tägliche Flexibilitätsbedarfe", fontsize=16)
ax.set_ylabel("GWh / Tag", fontsize=14)

# Move the legend outside the plot
ax.legend(
    loc="upper left",
    bbox_to_anchor=(1, 1),  # Position the legend outside
    title="Flexibilitätstechnologien",
)

# Tighten the layout to fit everything
plt.tight_layout()

# Display the plot
plt.show()

#### single

In [ ]:
# calc technology contribution for every region
result_single_all_buses_diff = {}

for bus_region in bus_regions_de:
    df_all_single = pd.DataFrame()
    for year in np.arange(2020, 2050, 5):
        df = result_single_all_buses[bus_region][year].copy()
        no_total_i = df.drop("no").index
        df[no_total_i] = -(df - df.loc["no"])[no_total_i]
        df_all_single = pd.concat([df_all_single, df], axis=1)

    result_single_all_buses_diff[bus_region] = df_all_single

# aggregate all regions
df_single_overall = pd.DataFrame(
    index=all_flex_techs, columns=years, dtype=float
).fillna(0.0)

for bus_region in bus_regions_de:
    df = result_single_all_buses_diff[bus_region]
    df = df.astype(float)
    df_single_overall.loc[df.index] = df_single_overall.loc[df.index].add(
        df, fill_value=0.0
    )

df_single_overall_annual_b = df_single_overall

In [ ]:
df = df_single_overall_annual_b.drop(["no"], errors="ignore")

# Set plot properties
fig, ax = plt.subplots(figsize=(8, 5))  # Increase figsize

# Extract colors for the bar plot
colors = [tech_colors[tech] for tech in df.index]

# Create the bar plot
df.T.plot(kind="bar", stacked=True, color=colors, ax=ax)

# Add title and labels
ax.set_title("Tägliche Flexibilitätsbedarfe", fontsize=16)
ax.set_ylabel("GWh / Tag", fontsize=14)

# Move the legend outside the plot
ax.legend(
    loc="upper left",
    bbox_to_anchor=(1, 1),  # Position the legend outside
    title="Flexibilitätstechnologien",
)

# Tighten the layout to fit everything
plt.tight_layout()

# Display the plot
plt.show()

## Testing & Plots

In [ ]:
n = networks[2045]
buses = ["DE0 20", "DE0 20 low voltage"]
electricity_supply, electricity_demand = supply_demand(
    n, interconnectors=True, merge_dist_grid=True, drop_dist_grid=False, buses=buses
)
vre_gens_s = list(set(vre_gens) & set(electricity_supply.index))
non_vre_gens = electricity_supply[~electricity_supply.index.isin(vre_gens)].index

vre_gen = electricity_supply.loc[vre_gens_s].sum()
load = electricity_demand.loc[load_carriers].sum()

In [ ]:
electricity_supply

In [ ]:
period = slice("2019-10-01", "2019-12-31")
tech1 = "AC"
tech2 = "coal"
tech3 = "lignite"
electricity_supply.loc[tech1][period].plot(figsize=(30, 5), label=tech1)
# electricity_supply.loc[tech2][period].plot(label=tech2)
# electricity_supply.loc[tech3][period].plot( label=tech3)
(load - vre_gen)[period].plot(label="load-vre_gen")
(load - vre_gen - electricity_supply.loc[tech1])[period].plot(
    label=f"load-vre_gen-{tech1}"
)
# (electricity_supply.loc["AC"] - electricity_demand.loc["AC"])[period].plot(label="AC balance")
# (-vre_gen)[period].plot(label="vre_gen")
plt.legend()

In [ ]:
period = slice("2019-01-01", "2019-03-01")
tech1 = "lignite"
tech2 = "coal"
tech3 = "lignite"
electricity_supply.loc[tech1][period].plot(figsize=(30, 5), label=tech1)
# electricity_supply.loc[tech2][period].plot(label=tech2)
# electricity_supply.loc[tech3][period].plot( label=tech3)
(load - vre_gen)[period].plot(label="load-vre_gen")
(load - vre_gen - electricity_supply.loc[tech1])[period].plot(
    label=f"load-vre_gen-{tech1}"
)
# (electricity_supply.loc["AC"] - electricity_demand.loc["AC"])[period].plot(label="AC balance")
# (-vre_gen)[period].plot(label="vre_gen")
plt.legend()

In [ ]:
n = networks[2020]
electricity_supply, electricity_demand = supply_demand(
    n, interconnectors=True, merge_dist_grid=True, drop_dist_grid=False
)

period = slice("2019-01-01", "2019-12-31")
# (demand-vre_gen)[period].plot(figsize=(30,5), label="demand-vre_gen")
# daily_average_hourly[period].plot(label="daily_average_hourly")
# weekly_average_hourly[period].plot(label="weekly_average_hourly")
monthly_average_hourly[period].plot(label="monthly_average_hourly")
yearly_average_hourly[period].plot(label="yearly_average_hourly")

plt.legend()

In [ ]:
period = slice("2019-01-01", "2019-03-01")
tech1 = "CCGT"
tech2 = "coal"
tech3 = "lignite"
electricity_supply.loc[tech1][period].plot(figsize=(30, 5), label=tech1)
electricity_supply.loc[tech2][period].plot(label=tech2)
electricity_supply.loc[tech3][period].plot(label=tech3)
(load - vre_gen)[period].plot(label="load-vre_gen")
# (demand-vre_gen)[period].plot(label="load-vre_gen")
# (-vre_gen)[period].plot(label="vre_gen")
plt.legend()

In [ ]:
# plot methodology
period = slice("2019-01-01", "2019-01-07")
vre_gen[period].plot(figsize=(20, 6), label="VRE generation")
load[period].plot(label="Load")
residual_load[period].plot(label="Residual load")
plt.legend()

In [ ]:
# plot methodology
period = slice("2019-01-01", "2019-01-07")
residual_load[period].plot(figsize=(20, 6), label="Residual load")
daily_average_residual[period].plot(label="Daily average residual load")
daily_average_hourly[period].plot(label="Daily average residual load (hourly)")
plt.legend()

In [ ]:
day = "2019-05-01"
(residual_load[day] / 1e3).plot(figsize=(20, 6), label="Residual load")
(daily_average_hourly[day] / 1e3).plot(label="Daily average residual load (hourly)")
plt.legend()

print(daily_flexibility_needs[day] / 1e3)
print(hourly_flexibility_needs[day] / 1e3)  # MWh

## Calc carrier contribution

In [ ]:
def calculate_technology_flexibility_contribution(
    n,
    bus_carrier=["AC", "low voltage"],
    load_carriers=["electricity", "agriculture electricity", "industry electricity"],
    vre_gens=[
        "onwind",
        "offwind-ac",
        "offwind-dc",
        "solar",
        "solar-hsat",
        "solar rooftop",
        "ror",
    ],
    region="DE",
    no_dist_grid=True,
):
    kwargs = {
        "groupby": n.statistics.groupers.get_name_bus_and_carrier,
        "nice_names": False,
    }
    buses_de = n.buses[
        (n.buses.index.str[:2] == region) & (n.buses.carrier.isin(bus_carrier))
    ].index

    electricity_supply = (
        n.statistics.supply(bus_carrier=bus_carrier, aggregate_time=False, **kwargs)
    ).multiply(n.snapshot_weightings.generators)
    electricity_supply = (
        electricity_supply[
            electricity_supply.index.get_level_values("bus").isin(buses_de)
        ]
        .groupby("carrier")
        .sum()
    )
    # electricity_supply = electricity_supply.drop(["AC", "DC", "electricity distribution grid"], errors="ignore")

    electricity_demand = (
        n.statistics.withdrawal(bus_carrier=bus_carrier, aggregate_time=False, **kwargs)
    ).multiply(n.snapshot_weightings.generators)
    electricity_demand = (
        electricity_demand[
            electricity_demand.index.get_level_values("bus").isin(buses_de)
        ]
        .groupby("carrier")
        .sum()
    )
    # electricity_demand = electricity_demand.drop(["AC", "DC", "electricity distribution grid"], errors="ignore")

    # assert (electricity_supply.values.sum() - electricity_demand.values.sum()) < 1e3, "Supply and demand must be equal"

    # 1. Calculate initial residual load
    vre_gen = electricity_supply.loc[vre_gens].sum()
    load = electricity_demand.loc[load_carriers].sum()
    demand = electricity_demand.sum()
    residual_load = load - vre_gen

    # 2. Compute daily average of residual load
    daily_average_residual = residual_load.resample("D").mean()

    # Expand daily averages to hourly resolution for comparison
    daily_average_hourly = daily_average_residual.reindex(
        residual_load.index, method="ffill"
    )

    # 3. Compute the difference between residual load and daily average (only the positive part)
    hourly_flexibility_needs = (residual_load - daily_average_hourly).clip(lower=0)

    # Calculate daily flexibility needs
    daily_flexibility_needs = hourly_flexibility_needs.resample("D").sum()

    # 4. Calculate total annual flexibility needs
    total_annual_flexibility = hourly_flexibility_needs.sum()

    # 5. Get supply technologies (excluding VRE)
    no_vre_supply = electricity_supply[~electricity_supply.index.isin(vre_gens)]

    assert (
        no_vre_supply.values.sum()
        + vre_gen.values.sum()
        - electricity_supply.values.sum()
    ) < 1e3, "Supply must be met by VRE and non VRE"

    # 6. Calculate individual technology contributions
    tech_contributions = {}

    for tech in no_vre_supply.index:
        # Create a copy of residual load
        residual_load_tech = residual_load.copy()

        # Subtract the specific technology's generation
        residual_load_tech -= no_vre_supply.loc[tech]

        # Recompute flexibility needs without this technology
        daily_average_residual_tech = residual_load_tech.resample("D").mean()
        daily_average_hourly_tech = daily_average_residual_tech.reindex(
            residual_load_tech.index, method="ffill"
        )

        # Compute hourly and daily flexibility needs
        hourly_flexibility_needs_tech = (
            residual_load_tech - daily_average_hourly_tech
        ).clip(lower=0)  # daily_average_hourly or with_tech
        daily_flexibility_needs_tech = hourly_flexibility_needs_tech.resample("D").sum()

        # Calculate the contribution of this technology
        tech_contribution = (
            total_annual_flexibility - daily_flexibility_needs_tech.sum()
        )
        tech_contributions[tech] = tech_contribution / 1e6  # Convert to TWh

    # Create a DataFrame of contributions
    residual_daily_flexibility_needs = pd.DataFrame.from_dict(
        tech_contributions, orient="index", columns=["flexibility"]
    )

    return {
        "total_annual_flexibility": total_annual_flexibility / 1e6,  # Convert to TWh
        "technology_contributions": residual_daily_flexibility_needs,
    }

In [ ]:
# Usage
year = 2045
n = networks[year]
result = calculate_technology_flexibility_contribution(n, region="DE")

# Print total and individual contributions
print("Total Annual Flexibility Needs:", result["total_annual_flexibility"], "TWh")
print("\nTechnology Contributions:")
print((result["technology_contributions"]).round(2))

# Verify additivity
print(
    "\nSum of Technology Contributions:",
    result["technology_contributions"]["flexibility"].sum(),
)
# print((result['technology_contributions'][result['technology_contributions'].flexibility > 0.1]).round(2).drop(["AC", "DC", "electricity distribution grid"]).sum())
print(result["technology_contributions"].round(2).sum())

In [ ]:
res = pd.DataFrame(index=np.arange(2020, 2050, 5), columns=["total_annual_flexibility"])
tech_contribution = {}
for year in np.arange(2020, 2050, 5):
    n = networks[year]
    result = calculate_technology_flexibility_contribution(n)
    res.loc[year, "total_annual_flexibility"] = result["total_annual_flexibility"]
    tech_contribution[year] = result["technology_contributions"]

In [ ]:
res

In [ ]:
all_years = summarize_small_values(tech_contribution[2020], "flexibility")
for year in np.arange(2025, 2050, 5):
    df = summarize_small_values(tech_contribution[year], "flexibility")
    all_years = pd.concat([all_years, df], axis=1)

In [ ]:
all_years

In [ ]:
all_years.columns = np.arange(2020, 2050, 5)
ax = all_years.T.plot(kind="bar", stacked=True, figsize=(10, 5), color=colors)

# Position the legend to the right of the plot
ax.legend(
    loc="center left", bbox_to_anchor=(1.0, 0.5), title="Flexibility technologies"
)  # Adjust as needed
ax.set_ylabel("Flexibility needs in TWh")  # Replace with your desired y-axis label

plt.tight_layout()  # Adjust layout to avoid clipping
plt.show()

In [ ]:
# calc flexibility contribution of tech
flexSign = (residual_load - daily_average_hourly).apply(
    lambda x: 1 if x > 0 else (-1 if x < 0 else 0)
)

In [ ]:
# tech
prod_tech = no_vre_supply.loc[tech]
daily_avg_prod_tech = prod_tech.resample("D").mean()
daily_avg_prod_tech = daily_avg_prod_tech.reindex(prod_tech.index, method="ffill")

con = 0.5 * flexSign * (prod_tech - daily_avg_prod_tech)

In [ ]:
def calculate_technology_flexibility_contribution_new(
    n,
    bus_carrier=["AC", "low voltage"],
    load_carriers=["electricity", "agriculture electricity", "industry electricity"],
    vre_gens=[
        "onwind",
        "offwind-ac",
        "offwind-dc",
        "solar",
        "solar-hsat",
        "solar rooftop",
        "ror",
    ],
):
    kwargs = {
        "groupby": n.statistics.groupers.get_name_bus_and_carrier,
        "nice_names": False,
    }
    buses_de = n.buses[
        (n.buses.index.str[:2] == region) & (n.buses.carrier.isin(bus_carrier))
    ].index

    electricity_supply = (
        n.statistics.supply(bus_carrier=bus_carrier, aggregate_time=False, **kwargs)
    ).multiply(n.snapshot_weightings.generators)
    electricity_supply = (
        electricity_supply[
            electricity_supply.index.get_level_values("bus").isin(buses_de)
        ]
        .groupby("carrier")
        .sum()
    )
    electricity_supply = electricity_supply.drop(
        ["AC", "DC", "electricity distribution grid"], errors="ignore"
    )

    electricity_demand = (
        n.statistics.withdrawal(bus_carrier=bus_carrier, aggregate_time=False, **kwargs)
    ).multiply(n.snapshot_weightings.generators)
    electricity_demand = (
        electricity_demand[
            electricity_demand.index.get_level_values("bus").isin(buses_de)
        ]
        .groupby("carrier")
        .sum()
    )
    electricity_demand = electricity_demand.drop(
        ["AC", "DC", "electricity distribution grid"], errors="ignore"
    )

    # 1. Calculate initial residual load
    vre_gen = electricity_supply.loc[vre_gens].sum()
    load = electricity_demand.loc[load_carriers].sum()
    residual_load = load - vre_gen

    # 2. Compute daily average of residual load
    daily_average_residual = residual_load.resample("D").mean()

    # Expand daily averages to hourly resolution for comparison
    daily_average_hourly = daily_average_residual.reindex(
        residual_load.index, method="ffill"
    )

    # 3. Compute the difference between residual load and daily average (only the positive part)
    hourly_flexibility_needs = (residual_load - daily_average_hourly).clip(lower=0)

    # Calculate daily flexibility needs
    daily_flexibility_needs = hourly_flexibility_needs.resample("D").sum()

    # 4. Calculate total annual flexibility needs
    total_annual_flexibility = hourly_flexibility_needs.sum()

    # 5. Get supply technologies (excluding VRE)
    no_vre_supply = electricity_supply[~electricity_supply.index.isin(vre_gens)]

    assert (
        no_vre_supply.values.sum()
        + vre_gen.values.sum()
        - electricity_supply.values.sum()
    ) < 1e3, "Supply must be met by VRE and non VRE"

    # 6. Calculate individual technology contributions
    flexSign = (residual_load - daily_average_hourly).apply(
        lambda x: 1 if x > 0 else (-1 if x < 0 else 0)
    )

    tech_contributions = {}
    for tech in no_vre_supply.index:
        # tech
        prod_tech = no_vre_supply.loc[tech]
        daily_avg_prod_tech = prod_tech.resample("D").mean()
        daily_avg_prod_tech = daily_avg_prod_tech.reindex(
            prod_tech.index, method="ffill"
        )

        tech_contribution = 0.5 * flexSign * (prod_tech - daily_avg_prod_tech)
        tech_contributions[tech] = tech_contribution.sum().sum() / 1e6  # Convert to TWh

    # Create a DataFrame of contributions
    residual_daily_flexibility_needs = pd.DataFrame.from_dict(
        tech_contributions, orient="index", columns=["flexibility"]
    )

    return {
        "total_annual_flexibility": total_annual_flexibility / 1e6,  # Convert to TWh
        "technology_contributions": residual_daily_flexibility_needs,
    }

In [ ]:
# Usage
year = 2045
n = networks[year]
result = calculate_technology_flexibility_contribution_new(n)

# Print total and individual contributions
print("Total Annual Flexibility Needs:", result["total_annual_flexibility"], "TWh")
print("\nTechnology Contributions:")
print((result["technology_contributions"]).round(2))

# Verify additivity
print(
    "\nSum of Technology Contributions:",
    result["technology_contributions"]["flexibility"].sum(),
)
# print((result['technology_contributions'][result['technology_contributions'].flexibility > 0.1]).round(2).drop(["AC", "DC", "electricity distribution grid"]).sum())
print(
    (
        result["technology_contributions"][
            result["technology_contributions"].flexibility > 0.1
        ]
    )
    .round(2)
    .sum()
)

In [ ]:
n = networks[2045]
bus_carrier = ["AC", "low voltage"]
load_carriers = ["electricity", "agriculture electricity", "industry electricity"]
vre_gens = [
    "onwind",
    "offwind-ac",
    "offwind-dc",
    "solar",
    "solar-hsat",
    "solar rooftop",
    "ror",
]
region = "DE"


kwargs = {
    "groupby": n.statistics.groupers.get_name_bus_and_carrier,
    "nice_names": False,
}
buses_de = n.buses[
    (n.buses.index.str[:2] == region) & (n.buses.carrier.isin(bus_carrier))
].index

electricity_supply = (
    n.statistics.supply(bus_carrier=bus_carrier, aggregate_time=False, **kwargs)
).multiply(n.snapshot_weightings.generators)
electricity_supply = (
    electricity_supply[electricity_supply.index.get_level_values("bus").isin(buses_de)]
    .groupby("carrier")
    .sum()
)
electricity_supply = electricity_supply.drop(
    ["AC", "DC", "electricity distribution grid"], errors="ignore"
)

electricity_demand = (
    n.statistics.withdrawal(bus_carrier=bus_carrier, aggregate_time=False, **kwargs)
).multiply(n.snapshot_weightings.generators)
electricity_demand = (
    electricity_demand[electricity_demand.index.get_level_values("bus").isin(buses_de)]
    .groupby("carrier")
    .sum()
)
electricity_demand = electricity_demand.drop(
    ["AC", "DC", "electricity distribution grid"], errors="ignore"
)

# assert (electricity_supply.values.sum() - electricity_demand.values.sum()) < 1e3, "Supply and demand must be equal"

# 1. Calculate initial residual load
vre_gen = electricity_supply.loc[vre_gens].sum()
load = electricity_demand.loc[load_carriers].sum()
demand = electricity_demand.sum()
residual_load = load - electricity_supply.sum()

# 2. Compute daily average of residual load
daily_average_residual = residual_load.resample("D").mean()

# Expand daily averages to hourly resolution for comparison
daily_average_hourly = daily_average_residual.reindex(
    residual_load.index, method="ffill"
)

# 3. Compute the difference between residual load and daily average (only the positive part)
hourly_flexibility_needs = (residual_load - daily_average_hourly).clip(lower=0)

# Calculate daily flexibility needs
daily_flexibility_needs = hourly_flexibility_needs.resample("D").sum()

# 4. Calculate total annual flexibility needs
total_annual_flexibility = hourly_flexibility_needs.sum()

round(hourly_flexibility_needs.sum() / 1e6, 2)

In [ ]:
# 1. Calculate initial residual load
vre_gen = electricity_supply.loc[vre_gens].sum()
load = electricity_demand.loc[load_carriers].sum()
demand = electricity_demand.sum()
residual_load = demand - vre_gen - electricity_supply.loc["battery discharger"]

In [ ]:
# 5. Get supply technologies (excluding VRE)
no_vre_supply = electricity_supply[~electricity_supply.index.isin(vre_gens)]

assert (
    no_vre_supply.values.sum() + vre_gen.values.sum() - electricity_supply.values.sum()
) < 1e3, "Supply must be met by VRE and non VRE"

# 6. Calculate individual technology contributions
tech_contributions = {}

for tech in ["nuclear"]:
    # Subtract the specific technology's generation
    residual_load_tech = residual_load - no_vre_supply.loc[tech]

    # Recompute flexibility needs without this technology
    daily_average_residual_tech = residual_load_tech.resample("D").mean()
    daily_average_hourly_tech = daily_average_residual_tech.reindex(
        residual_load_tech.index, method="ffill"
    )

    # Compute hourly and daily flexibility needs
    hourly_flexibility_needs_tech = (
        residual_load_tech - daily_average_hourly_tech
    ).clip(lower=0)  # daily_average_hourly or with_tech
    daily_flexibility_needs_tech = hourly_flexibility_needs_tech.resample("D").sum()

    # Calculate the contribution of this technology
    tech_contribution = total_annual_flexibility - daily_flexibility_needs_tech.sum()
    tech_contributions[tech] = tech_contribution / 1e6  # Convert to TWh

# Create a DataFrame of contributions
residual_daily_flexibility_needs = pd.DataFrame.from_dict(
    tech_contributions, orient="index", columns=["flexibility"]
)

In [ ]:
residual_daily_flexibility_needs.sort_values(by="flexibility", ascending=False).head(10)

In [ ]:
tech

In [ ]:
total_annual_flexibility / 1e6

In [ ]:
daily_flexibility_needs_tech.sum() / 1e6

In [ ]:
# 1. Calculate initial residual load
vre_gen = electricity_supply.loc[vre_gens].sum()
load = electricity_demand.loc[load_carriers].sum()
demand = electricity_demand.sum()
residual_load = load - vre_gen

# 2. Compute daily average of residual load
daily_average_residual = residual_load.resample("D").mean()

# Expand daily averages to hourly resolution for comparison
daily_average_hourly = daily_average_residual.reindex(
    residual_load.index, method="ffill"
)

# 3. Compute the difference between residual load and daily average (only the positive part)
hourly_flexibility_needs = (residual_load - daily_average_hourly).clip(lower=0)

# Calculate daily flexibility needs
daily_flexibility_needs = hourly_flexibility_needs.resample("D").sum()

# 4. Calculate total annual flexibility needs
total_annual_flexibility = hourly_flexibility_needs.sum()

# 5. Get supply technologies (excluding VRE)
no_vre_supply = electricity_supply[~electricity_supply.index.isin(vre_gens)]

round(total_annual_flexibility / 1e6, 2)

## Flexibilität

In [ ]:
network = networks[2045]
ct = "DE"
buses = network.buses.index[(network.buses.index.str[:2] == ct)].drop("DE")
balance = (
    network.statistics.energy_balance(
        aggregate_time=False,
        nice_names=False,
        groupby=network.statistics.groupers.get_bus_and_carrier_and_bus_carrier,
    )
    .loc[:, buses, :, :]
    .droplevel("bus")
)

In [ ]:
period = slice("2019-01-18", "2019-01-25")
carriers = ["AC", "low voltage"]
mask = balance.index.get_level_values("bus_carrier").isin(carriers)
nb = balance[mask].groupby("carrier").sum().div(1e3).T.loc[period]

In [ ]:
# nb.sum().sort_values(ascending=False).head(20)

In [ ]:
# nb.sum().sort_values(ascending=False).tail(20)

In [ ]:
period = slice("2019-01-11", "2019-01-18")
carriers = ["AC", "low voltage"]
mask = balance.index.get_level_values("bus_carrier").isin(carriers)
nb = balance[mask].groupby("carrier").sum().div(1e3).T.loc[period]

In [ ]:
# nb.sum().sort_values(ascending=False).tail(20)

In [ ]:
# installed capacities

In [ ]:
# Electricity supply
# electricity generation DE 2020: 488 TWh
# if aggregate_time=False the results are in MW (you need to multiply with snapshot.weightings)

region = "DE"
n = networks[2020]

kwargs = {
    "groupby": n.statistics.groupers.get_name_bus_and_carrier,
    "nice_names": False,
}

electricity_cap = (
    n.statistics.supply(bus_carrier=["low voltage", "AC"], **kwargs)
    .filter(like=region)
    .groupby(["carrier"])
    .sum()
)

In [ ]:
n = networks[2045]
s, d = supply_demand(
    n, interconnectors=True, merge_dist_grid=False, drop_dist_grid=False
)

In [ ]:
s.loc["DC"].resample("D").mean().plot()

In [ ]:
# electricity import + exports
ct = "DE"
incoming_line = n.lines.index[
    (n.lines.carrier == "AC")
    & (n.lines.bus0.str[:2] != ct)
    & (n.lines.bus1.str[:2] == ct)
]
outgoing_line = n.lines.index[
    (n.lines.carrier == "AC")
    & (n.lines.bus0.str[:2] == ct)
    & (n.lines.bus1.str[:2] != ct)
]

incoming_link = n.links.index[
    (n.links.carrier == "DC")
    & (n.links.bus0.str[:2] != ct)
    & (n.links.bus1.str[:2] == ct)
]
outgoing_link = n.links.index[
    (n.links.carrier == "DC")
    & (n.links.bus0.str[:2] == ct)
    & (n.links.bus1.str[:2] != ct)
]

In [ ]:
df = (
    (
        -(
            n.lines_t.p0[outgoing_line].sum(axis=1)
            + n.links_t.p0[outgoing_link].sum(axis=1)
        )
        - (
            n.lines_t.p0[incoming_line].sum(axis=1)
            + n.links_t.p0[incoming_link].sum(axis=1)
        )
    )
    * 3
    / 1e3
)
df.plot()

In [ ]:
df * 3 / 1e6